# Classification Machine-learning des APP

## Import des packages et données
Variables créées : 
- df = dataframe initial

In [ ]:
# Import libraries
import pandas as pd # manipulation de donnees
import numpy as np # calculs numeriques
import matplotlib.pyplot as plt # graphiques
import seaborn as sns # visualisation avancee
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
# StandardScaler : normalise les caracteristiques numeriques pour qu’elles aient moyenne 0 et variance 1.
# LabelEncoder : convertit des etiquettes categorielles (texte) en entiers (0, 1, 2…).
# classification_report : resume (texte) des metriques par classe : precision (precision), rappel (recall), F1-score, support (taille de classe).
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, 
                            accuracy_score, f1_score)
from sklearn.model_selection import (train_test_split, StratifiedKFold, 
                                    cross_val_score, RepeatedStratifiedKFold)
from sklearn.pipeline import Pipeline # Pipeline : enchaene proprement des etapes
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# Feature selection
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

# Imputation
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Base classes
from sklearn.base import BaseEstimator, TransformerMixin

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# Load and initial exploration
df = pd.read_excel("data.xlsx",engine='openpyxl')
print(f"Dataset shape: {df.shape}") # Taille du jeu de donnees
print(f"\nTarget distribution:") # Distribution de la cible
print(df['Phenotype_final'].value_counts()) # Detecter un desequilibre de classes qui influence le choix des metriques (par ex. F1‑score plutôt qu’accuracy) et des techniques (ponderation des classes, reechantillonnage).
print(f"\nMissing values per column:") # Pour voir les donnees manquantes par colonne
print(df.isnull().sum().sort_values(ascending=False).head(10))

## 1. Data Preprocessing Classes (imputation de données)

### 1.1. Imputations multiples MICE

Variables créées :
- **ImputationPipeline** = ensemble de tâches/paramètres définis pour imputer les données manquantes (gérer les données manquantes)
- init () = fonction pour définir les groupes de variables pour l'imputation
_Certains groupes ont été changés car les corrélations étaient basses_
- fit () = fonction pour définir les paramètres de l'imputation
- self = groupes de variables définis a priori
- group_name = noms des groupes de variables
- feature_groups = dictionnaire des groupes de variables
- single_features = variables imputées individuellement
- **X_df** = nouveau dataframe pour appliquer le pipeline de l'imputation
    > Paramètres particuliers : arrondir à des nombres entiers + Narrative_Discourse_Total; Reading_Words; Writing_Dictation_Words considérés comme totaux de sous-scores.

#### 1.1.1. Création du Pipeline et application

Ajouts/modifications effectué.e.s par rapport au notebook d'origine :
- Changé quelques groupes d'imputations
- Arrondir pour obtenir de nombres entiers comme données imputées 
- Variables composée : score total = addition des sous-scores (exemple pour narrative_discourse il y a _total = _Actions + _Recit + etc.)

In [ ]:
class ImputationPipeline(BaseEstimator, TransformerMixin): # cette classe est conçue pour etre un transformateur scikit‑learn compatible avec les pipelines
    """Handles missing value imputation for different variable groups"""
    
    def __init__(self): # A l’entrainement, on ajuste un imputeur pour chaque groupe de colonnes (mais chaque colonne est imputée indépendamment). A la transformation, on applique l’imputeur du groupe a ses colonnes respectives.
        self.imputers_ = {}
        self.feature_groups_ = { # Pas besoin de mettre un groupe pour l'entretien dirige, il n'y a pas de donnees manquantes
            'repetition': ['Repetition_Words', 'Repetition_Sentence', 'Repetition_NonWords'],
            'fluency': ['Fluency_Verbs', 'Fluency_Semantic', 'Fluency_Literal'],
            'denomination': ['Denomination_Objects', 'Denomination_Verbs', 'Denomination_FamousPeople'],
            'lexic':['Anamnestic_ITW_Lexic','Anamnestic_ITW_Info','Narrative_Discourse_Lexic','Narrative_Discourse_Info'],
            'syntax_production': ['Elaboration_Sentences', 'Narrative_Discourse_Syntax','Anamnestic_ITW_Syntax'],
            'writing_words': ['Writing_Dictation_Words', 'Writing_Dictation_Reg_Words', 'Writing_Dictation_Irreg_Words', 'Writing_Dictation_Sentences'],
            'reading_words': ['Reading_Words', 'Reading_Reg_Words','Reading_Irreg_Words'],
            'Nonwords': ['Reading_NonWords', 'Writing_Dictation_Nonwords', 'Writing_Dictation_Sentences'], # Mis écriture phrases car corrélation 0.56 avec écriture non mots 
            'syntax_comprehension': ['Execution_Commands','Understanding_Syntax'],
            'association': ['Association_Photo_w_OralWord', 'Association_WritenWord_w_Photo'],
            'narrative_discourse':['Narrative_Discourse_Total','Narrative_Discourse_Actions','Narrative_Discourse_Recit'],
            'pragma':['Narrative_Discourse_Pragma','Anamnestic_ITW_Pragmatics']
        }
        
    def fit(self, X, y=None):
        X_df = X.copy() # Copie des donnees pour eviter de modifier l'original
        
        # Fit imputers for each group
        for group_name, features in self.feature_groups_.items(): # Filtrage des colonnes, previent si une attendue absente
            available_features = [f for f in features if f in X_df.columns]
            not_available_features = [f for f in features if f not in X_df.columns]
            if not_available_features:
                print(f"Warning: Features {not_available_features} not found in the dataset for group '{group_name}'")
            if len(available_features) > 1: # Principe du chained equations ou MICE, colonne avec NA regressee avec les autres colonnes avec iterations jusqu'a stabilisation
                imputer = IterativeImputer(
                    estimator=LinearRegression(),
                    max_iter=10,
                    random_state=42,
                    min_value=0
                )
                self.imputers_[group_name] = imputer
                self.imputers_[group_name].fit(X_df[available_features])
        
        # Handle single features with median imputation (strategie d'imputation)
        self.medians_ = {}
        single_features = ['Anamnestic_ITW_Attention', 'Writing_Automatic', 'Understanding_Text']
        for feature in single_features:
            if feature in X_df.columns:
                self.medians_[feature] = X_df[feature].median()
        
        return self
    
    def transform(self, X):
        X_df = X.copy()
        
        # Apply group imputation
        # Si on a bien un imputeur entraine pour ce groupe, et qu’il y a ≥ 2 colonnes presentes, on applique transform qui remplit les NA en s’aidant des autres colonnes du groupe, selon les relations apprises pendant fit.
        for group_name, features in self.feature_groups_.items():
            available_features = [f for f in features if f in X_df.columns]
            if group_name in self.imputers_ and len(available_features) > 1:
                X_df[available_features] = self.imputers_[group_name].transform(X_df[available_features])
        
        # Apply median imputation for single features
        for feature, median_val in self.medians_.items():
            if feature in X_df.columns:
                X_df[feature].fillna(median_val, inplace=True)
        
        # Arrondir tous les nombres à des entiers
        # Convertir les colonnes numeriques en nombres entiers (arrondi standard)
        numeric_cols = X_df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            X_df[col] = X_df[col].round().astype('int')  # Convertir en entiers avec arrondi
        
        # ============================================================================
        # Correction des variables composées (garantir cohérence composée = somme composantes)
        # ============================================================================
        
        # 1️⃣ Narrative_Discourse_Total = sum of 6 components
        narrative_components = [
            'Narrative_Discourse_Actions',
            'Narrative_Discourse_Lexic',
            'Narrative_Discourse_Syntax',
            'Narrative_Discourse_Recit',
            'Narrative_Discourse_Info',
            'Narrative_Discourse_Pragma'
        ]
        narrative_available = [col for col in narrative_components if col in X_df.columns]
        if 'Narrative_Discourse_Total' in X_df.columns and len(narrative_available) == 6:
            X_df['Narrative_Discourse_Total'] = X_df[narrative_available].sum(axis=1).astype('int')
        
        # 2️⃣ Reading_Words = Reading_Reg_Words + Reading_Irreg_Words
        reading_components = ['Reading_Reg_Words', 'Reading_Irreg_Words']
        reading_available = [col for col in reading_components if col in X_df.columns]
        if 'Reading_Words' in X_df.columns and len(reading_available) == 2:
            X_df['Reading_Words'] = X_df[reading_available].sum(axis=1).astype('int')
        
        # 3️⃣ Writing_Dictation_Words = Writing_Dictation_Reg_Words + Writing_Dictation_Irreg_Words
        writing_components = ['Writing_Dictation_Reg_Words', 'Writing_Dictation_Irreg_Words']
        writing_available = [col for col in writing_components if col in X_df.columns]
        if 'Writing_Dictation_Words' in X_df.columns and len(writing_available) == 2:
            X_df['Writing_Dictation_Words'] = X_df[writing_available].sum(axis=1).astype('int')

        return X_df


class Residualizer(BaseEstimator, TransformerMixin):
    """Removes confounding effects using linear regression residuals"""
    
    def __init__(self, confounders, drop_confounders=True):
        self.confounders = confounders if isinstance(confounders, list) else [confounders]
        self.drop_confounders = drop_confounders
        self.models_ = {}
        
    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Fit regression models for each numeric feature
        numeric_cols = X_df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if col not in self.confounders:
                model = LinearRegression()
                model.fit(X_df[self.confounders], X_df[col])
                self.models_[col] = model
        
        return self
    
    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Apply residualization
        for col, model in self.models_.items():
            if col in X_df.columns:
                predictions = model.predict(X_df[self.confounders])
                X_df[col] = X_df[col] - predictions
        
        # Optionally drop confounders
        if self.drop_confounders:
            X_df = X_df.drop(columns=self.confounders, errors='ignore')
        
        return X_df # On renvoie le dataframe impute

#### 1.1.2. Représentations graphiques

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

# Ce bloc est pour les representations graphiques on va voir :
# Repartition des donnees originales completes (nuage bleu).
# Emplacements des points imputes (nuage rouge) dans le plan 2D des paires
# Segments indiquant le deplacement entre le point original (quand une des deux composantes etait connue) et le point impute, pour voir si l’imputation reste localement coherente.
# Pour la visualisation, reentrainement d'un imputeur multivarie par groupe et comparaison visuelle des donnees originales vs imputees

# Your feature groups
feature_groups = {
    'repetition': ['Repetition_Words', 'Repetition_Sentence', 'Repetition_NonWords'],
    'fluency': ['Fluency_Verbs', 'Fluency_Semantic', 'Fluency_Literal'],
    'denomination': ['Denomination_Objects', 'Denomination_Verbs', 'Denomination_FamousPeople'],
    'lexic':['Anamnestic_ITW_Lexic','Anamnestic_ITW_Info','Narrative_Discourse_Lexic','Narrative_Discourse_Info'],
    'syntax_production': ['Elaboration_Sentences', 'Narrative_Discourse_Syntax','Anamnestic_ITW_Syntax'],
    'writing_words': ['Writing_Dictation_Words', 'Writing_Dictation_Reg_Words', 'Writing_Dictation_Irreg_Words', 'Writing_Dictation_Sentences'],
    'reading_words': ['Reading_Words', 'Reading_Reg_Words','Reading_Irreg_Words'],
    'Nonwords': ['Reading_NonWords', 'Writing_Dictation_Nonwords', 'Writing_Dictation_Sentences'], # Mis écriture phrases car corrélation 0.56 avec écriture non mots 
    'syntax_comprehension': ['Execution_Commands','Understanding_Syntax'],
    'association': ['Association_Photo_w_OralWord', 'Association_WritenWord_w_Photo'],
    'narrative_discourse':['Narrative_Discourse_Total','Narrative_Discourse_Actions','Narrative_Discourse_Recit'],
    'pragma':['Narrative_Discourse_Pragma','Anamnestic_ITW_Pragmatics']
}

# Plot pairwise for group
def plot_imputation_for_group(df, group_name, features): # Dataframe avec donnees imputees, tracer des nuages de points
    print(f"\n=== Group: {group_name} ===")
    available = [f for f in features if f in df.columns]
    if len(available) < 2:
        print(f"Skipping group '{group_name}': < 2 available features.")
        return
    
    imputer = IterativeImputer(estimator=LinearRegression(), random_state=0)
    imputed_data = imputer.fit_transform(df[available])
    imputed_df = pd.DataFrame(imputed_data, columns=available)
    
    for i in range(len(available)): # Boucle de visualisation paire par paire
        for j in range(i+1, len(available)):
            x_col = available[i]
            y_col = available[j]
            print(f"Plotting: {x_col} vs {y_col}")
            
            data = df[[x_col, y_col]].copy()
            complete_mask = data.notnull().all(axis=1)
            incomplete_mask = ~complete_mask

            n_original = complete_mask.sum()
            n_imputed = incomplete_mask.sum()

            
            plt.figure(figsize=(8,6)) # Points bleus pour les couples originaux pour les lignes completes ; points rouges pour les couples imputes
            plt.scatter(data.loc[complete_mask, x_col], data.loc[complete_mask, y_col],
                        label='Original Data', alpha=0.7, color='blue')
            
            plt.scatter(imputed_df.loc[incomplete_mask, x_col], imputed_df.loc[incomplete_mask, y_col],
                        label='Imputed Data', alpha=0.7, color='red', marker='x')
            
            for idx in data.index[incomplete_mask]:
                original_x = data.loc[idx, x_col]
                original_y = data.loc[idx, y_col]
                imputed_x = imputed_df.loc[idx, x_col]
                imputed_y = imputed_df.loc[idx, y_col]
                
                if pd.isna(original_x) and not pd.isna(original_y):
                    # x manquait: on trace un segment vertical entre y original et y impute (meme x impute)
                    plt.plot([imputed_x, imputed_x], [original_y, imputed_y], color='gray', linestyle='dashed', alpha=0.7)
                elif pd.isna(original_y) and not pd.isna(original_x):
                    # y manquait: segment horizontal entre x original et x impute (meme y impute)
                    plt.plot([original_x, imputed_x], [imputed_y, imputed_y], color='gray', linestyle='dashed', alpha=0.7)
                elif pd.isna(original_x) and pd.isna(original_y):
                    # les deux manquaient: pas de segment (on ne peut pas relier a un original)
                    pass
            
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f"{group_name}: {y_col} vs {x_col} (Iterative Imputation)")
            plt.legend()
            plt.grid(True)
            plt.show()

# Example use: loop on your groups
for group_name, features in feature_groups.items():
    plot_imputation_for_group(df, group_name, features)


### 1.2. Corrélations et rapport d'imputation

Corrélations de Pearson sans vérification des conditions d'application et si significatives. Pour avoir une idée de si les groupes effectués pour les imputations sont pertinents.

In [ ]:
# Correlations entre les variables des groupes

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Heatmap de correlation par groupe
def plot_correlation_heatmaps_by_group(
    df: pd.DataFrame,
    feature_groups: dict,
    save_dir: str | None = None,
    figsize=(8, 6),
    annot: bool = True,
    corr_method: str = "pearson",
):
    """
    Trace (et eventuellement enregistre) une heatmap de correlations pour chaque groupe.
    - df : DataFrame d'entrée
    - feature_groups : dict {nom_groupe: [colonnes]}
    - save_dir : dossier ou enregistrer les figures (None => affiche simplement)
    - figsize : taille des figures
    - annot : afficher les coefficients chiffres sur la heatmap
    - corr_method : 'pearson' (par défaut), 'spearman' (si non-linéarités), etc.
    """
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    for group, cols in feature_groups.items():
        # On ne garde que les colonnes numeriques existantes
        available = [c for c in cols if c in df.columns]
        num_cols = df[available].select_dtypes(include=[np.number])
        if num_cols.shape[1] < 2:
            print(f"[Heatmap] Groupe '{group}': < 2 colonnes numériques disponibles. Skip.")
            continue

        # Correlation pairwise (pandas ignore les NA pairwise)
        corr = num_cols.corr(method=corr_method)

        plt.figure(figsize=figsize)
        sns.heatmap(
            corr,
            vmin=-1,
            vmax=1,
            cmap="coolwarm",
            annot=annot,
            fmt=".2f",
            square=True,
            cbar_kws={"label": "Corrélation"},
        )
        plt.title(f"Corrélations – {group} (méthode: {corr_method})", fontsize=13, fontweight='bold')
        plt.tight_layout()

        if save_dir:
            path = os.path.join(save_dir, f"heatmap_correlation_{group}.png")
            plt.savefig(path, dpi=150)
            plt.close()
            print(f"[Heatmap] Enregistrée: {path}")
        else:
            plt.show()


# Executions et visualisations

print("="*70)
print("1️⃣ HEATMAPS DE CORRÉLATIONS PAR GROUPE")
print("="*70)

# Tracer les heatmaps de corrélations pour chaque groupe
plot_correlation_heatmaps_by_group(df, feature_groups, annot=True, corr_method="pearson")

print("\n" + "="*70)
print("2️⃣ RAPPORT D'IMPUTATION DÉTAILLÉ PAR COLONNE")
print("="*70)


### 1.3. Evaluation des imputations

Analyse synthétique de la qualité et de la fiabilité de l'imputation des données manquantes : z-score, écart absolu à la médiane, analyse de sensibilité

- ⚠️ IMPORTANT : imputation GLOBALE (fit sur l'ensemble du dataframe, utile UNIQUEMENT pour le rapport de qualité d'imputation ci-dessous (z-scores, sensibilité, sur toute la cohorte).
- ⚠️ NE PAS utiliser df_after pour la PCA / le positionnement des individus supplémentaires : cela réintroduirait une fuite de données (l'imputeur aurait "vu" les individus test et supplémentaires au moment de son ajustement).
- ➜ Pour la PCA, utilisez df_imputed_no_leak, calculé plus bas dans la section "Imputation sans fuite de données"

#### 1.3.1. Résumé des imputations et écarts

Variables créées pour générer les rapports : 
- df_before = copie dataframe d'origine sans imputations
- df_after = dataframe avec application d'ImputationPipeline

In [ ]:
# ============================================================================
# PRÉPARATION DE L'IMPUTATION : Sauvegarde avant/après et fonction de rapport
# ============================================================================

# Sauvegarder une copie des données AVANT imputation
df_before = df.copy()

# Appliquer le pipeline d'imputation
imputation_pipeline = ImputationPipeline()
imputation_pipeline.fit(df_before)
df_after = imputation_pipeline.transform(df_before)

print(f"✅ Pipeline d'imputation appliqué")
print(f"   AVANT: {df_before.shape} | Valeurs manquantes: {df_before.isna().sum().sum()}")
print(f"   APRÈS: {df_after.shape} | Valeurs manquantes: {df_after.isna().sum().sum()}")

# ============================================================================
# FONCTION DE RAPPORT D'IMPUTATION PAR COLONNE
# ============================================================================

def imputation_report_by_column(df_before, df_after, feature_groups):
    """
    Génère un rapport détaillé des imputations par colonne.
    Compare les données avant et après imputation pour évaluer la qualité.
    """
    report_data = []
    
    for col in df_before.columns:
        # Compter les valeurs manquantes
        n_missing_before = df_before[col].isna().sum()
        n_missing_after = df_after[col].isna().sum() if col in df_after.columns else 0
        n_imputed = n_missing_before - n_missing_after
        pct_imputed = (n_imputed / len(df_before) * 100) if len(df_before) > 0 else 0
        
        # Statistiques sur les valeurs imputées
        if n_imputed > 0:
            # Créer un masque pour les valeurs imputées
            mask_imputed = df_before[col].isna() & df_after[col].notna()
            imputed_values = df_after.loc[mask_imputed, col]
            
            # Calculer les statistics (seulement si colonne numérique)
            if pd.api.types.is_numeric_dtype(imputed_values):
                imputed_median = imputed_values.median() if len(imputed_values) > 0 else np.nan
                imputed_mean = imputed_values.mean() if len(imputed_values) > 0 else np.nan
                
                # Z-scores par rapport à la distribution originale
                if df_before[col].notna().sum() > 0:
                    orig_mean = df_before[col].mean()
                    orig_std = df_before[col].std()
                    if orig_std > 0:
                        z_scores = np.abs((imputed_values - orig_mean) / orig_std)
                        imputed_zscore_abs_mean = z_scores.mean()
                    else:
                        imputed_zscore_abs_mean = 0
                else:
                    imputed_zscore_abs_mean = np.nan
                
                # Écart à la médiane originale
                if df_before[col].notna().sum() > 0:
                    orig_median = df_before[col].median()
                    abs_dev = np.abs(imputed_values - orig_median)
                    imputed_abs_dev_from_original_median_mean = abs_dev.mean()
                else:
                    imputed_abs_dev_from_original_median_mean = np.nan
            else:
                imputed_median = np.nan
                imputed_mean = np.nan
                imputed_zscore_abs_mean = np.nan
                imputed_abs_dev_from_original_median_mean = np.nan
        else:
            imputed_median = np.nan
            imputed_mean = np.nan
            imputed_zscore_abs_mean = np.nan
            imputed_abs_dev_from_original_median_mean = np.nan
        
        # Trouver le groupe
        group = None
        for grp_name, features in feature_groups.items():
            if col in features:
                group = grp_name
                break
        
        # Vérifier l'intégrité (les valeurs non-manquantes ne doivent pas changer)
        mask_non_missing = df_before[col].notna()
        if mask_non_missing.sum() > 0 and pd.api.types.is_numeric_dtype(df_before[col]):
            try:
                before_values = df_before.loc[mask_non_missing, col].sum()
                after_values = df_after.loc[mask_non_missing, col].sum() if col in df_after.columns else 0
                delta = abs(before_values - after_values)
            except:
                delta = 0
        else:
            delta = 0
        
        # Ajouter au rapport
        report_data.append({
            'colonne': col,
            'groupe': group,
            'nb_observations': len(df_before),
            'n_missing_before': n_missing_before,
            'n_imputed': n_imputed,
            'pct_imputed': pct_imputed,
            'imputed_median': imputed_median,
            'imputed_mean': imputed_mean,
            'imputed_zscore_abs_mean': imputed_zscore_abs_mean,
            'imputed_abs_dev_from_original_median_mean': imputed_abs_dev_from_original_median_mean,
            'delta_non_missing_abs_sum': delta
        })
    
    return pd.DataFrame(report_data)

# Générer le rapport d'imputation
imputation_report = imputation_report_by_column(df_before, df_after, feature_groups)
print(f"\n✅ Rapport d'imputation généré : {len(imputation_report)} colonnes analysées")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("🔍 ÉVALUATION SYNTHÉTIQUE DES IMPUTATIONS".center(80))
print("="*80)

# Vérifier que le rapport d'imputation existe
if 'imputation_report' not in dir():
    print("⚠️ Rapport d'imputation non disponible. Génération du rapport...")
    # Régénérer si nécessaire
    try:
        imputation_report = imputation_report_by_column(df_before, df_after, feature_groups)
    except:
        print("❌ Impossible de générer le rapport.")

# ============================================================================
# 1️⃣ RÉSUMÉ GLOBAL
# ============================================================================
print("\n" + "█"*80)
print("1️⃣  RÉSUMÉ GLOBAL DES IMPUTATIONS")
print("█"*80)

total_cells = df_before.shape[0] * df_before.shape[1]
total_missing_before = df_before.isna().sum().sum()
total_missing_after = df_after.isna().sum().sum()
total_imputed = total_missing_before - total_missing_after
pct_of_total = (total_imputed / total_cells * 100)

print(f"\n📊 Couverture des données :")
print(f"   • Nombre total de cellules : {total_cells:,}")
print(f"   • Valeurs manquantes AVANT : {total_missing_before:,} ({total_missing_before/total_cells*100:.2f}%)")
print(f"   • Valeurs manquantes APRÈS : {total_missing_after:,} ({total_missing_after/total_cells*100:.2f}%)")
print(f"   • Valeurs IMPUTÉES : {total_imputed:,} ({pct_of_total:.2f}%)")
print(f"   • Réduction des NA : {(total_missing_before-total_missing_after)/total_missing_before*100:.1f}%")

# ============================================================================
# 2️⃣ TOP COLONNES IMPUTÉES
# ============================================================================
print("\n" + "█"*80)
print("2️⃣  TOP 15 COLONNES AVEC LE PLUS D'IMPUTATIONS")
print("█"*80)

top_imputed = imputation_report[imputation_report['n_imputed'] > 0].head(15)
print(f"\n{'Colonne':<40} {'Imputées':<12} {'%':<8} {'Médiane':<12} {'Z-score':<10}")
print("-"*80)
for idx, row in top_imputed.iterrows():
    col_name = row['colonne'][:38]
    n_imp = int(row['n_imputed'])
    pct = row['pct_imputed']
    med = row['imputed_median'] if pd.notna(row['imputed_median']) else np.nan
    zscore = row['imputed_zscore_abs_mean'] if pd.notna(row['imputed_zscore_abs_mean']) else np.nan
    
    print(f"{col_name:<40} {n_imp:<12} {pct:>6.1f}% {med:>12.2f} {zscore:>10.2f}")

# ============================================================================
# 3️⃣ QUALITÉ DES IMPUTATIONS PAR INDICATEUR
# ============================================================================
print("\n" + "█"*80)
print("3️⃣  QUALITÉ DES IMPUTATIONS (Analyse Statistique)")
print("█"*80)

imputed_cols = imputation_report[imputation_report['n_imputed'] > 0].copy()

if len(imputed_cols) > 0:
    # Z-scores (éloignement de la distribution originale)
    zscore_mean = imputed_cols['imputed_zscore_abs_mean'].mean()
    zscore_median = imputed_cols['imputed_zscore_abs_mean'].median()
    zscore_std = imputed_cols['imputed_zscore_abs_mean'].std()
    
    # Écarts à la médiane
    mad_mean = imputed_cols['imputed_abs_dev_from_original_median_mean'].mean()
    mad_median = imputed_cols['imputed_abs_dev_from_original_median_mean'].median()
    
    # % de colonnes avec données cohérentes (Z-score < 2)
    coherent_count = (imputed_cols['imputed_zscore_abs_mean'] < 2).sum()
    coherent_pct = (coherent_count / len(imputed_cols)) * 100
    
    print(f"\n📈 Z-scores absolus (mesure d'écart à la distribution) :")
    print(f"   • Moyenne : {zscore_mean:.3f}")
    print(f"   • Médiane : {zscore_median:.3f}")
    print(f"   • Écart-type : {zscore_std:.3f}")
    print(f"   → Interprétation : Z < 1 = très proche ; Z < 2 = acceptable ; Z > 3 = anomalie")
    print(f"   → {coherent_count} colonnes ({coherent_pct:.1f}%) ont Z-score < 2 ✓")
    
    print(f"\n📏 Écart absolu à la médiane originale :")
    print(f"   • Moyenne : {mad_mean:.3f}")
    print(f"   • Médiane : {mad_median:.3f}")
    print(f"   → Mesure de plausibilité : écart avec le centre de la distribution")
    
    # Cohérence des données originales
    delta_mean = imputation_report['delta_non_missing_abs_sum'].mean()
    print(f"\n✅ Intégrité des données originales :")
    print(f"   • Écart somme des non-manquantes : {delta_mean:.6f}")
    print(f"   → (Doit être ~0 pour vérifier que les valeurs observées n'ont pas changé)")

# ============================================================================
# 4️⃣ ANALYSE PAR GROUPE
# ============================================================================
print("\n" + "█"*80)
print("4️⃣  RÉSUMÉ PAR GROUPE DE VARIABLES")
print("█"*80)

if 'groupe' in imputation_report.columns:
    group_summary = imputation_report.groupby('groupe', dropna=False).agg({
        'colonne': 'count',
        'n_imputed': 'sum',
        'pct_imputed': 'mean',
        'imputed_zscore_abs_mean': 'mean',
        'imputed_abs_dev_from_original_median_mean': 'mean'
    }).sort_values('n_imputed', ascending=False)
    
    group_summary.columns = ['Nb_Colonnes', 'Total_Imputees', 'Pct_Moy', 'Z-score_Moy', 'Ecart_Med_Moy']
    
    print("\n" + group_summary.to_string())
    
    print(f"\n🎯 Groupe avec le plus d'imputations :")
    if len(group_summary) > 0:
        top_group = group_summary.index[0]
        top_count = int(group_summary.iloc[0]['Total_Imputees'])
        print(f"   → {top_group} : {top_count} valeurs imputées")

# ============================================================================
# 5️⃣ INDICATEURS DE RISQUE
# ============================================================================
print("\n" + "█"*80)
print("5️⃣  INDICATEURS DE RISQUE / ANOMALIES")
print("█"*80)

anomalies = imputed_cols[imputed_cols['imputed_zscore_abs_mean'] > 3]
print(f"\n⚠️ Colonnes avec Z-score > 3 (très éloignées de la distribution) :")
if len(anomalies) > 0:
    for idx, row in anomalies.iterrows():
        print(f"   ❌ {row['colonne']} : Z-score = {row['imputed_zscore_abs_mean']:.3f}")
else:
    print(f"   ✅ Aucune anomalie détectée")

# Colonnes avec très peu de données originales
sparse = imputation_report[(imputation_report['n_missing_before'] > imputation_report['nb_observations'] * 0.5) & 
                           (imputation_report['n_imputed'] > 0)]
print(f"\n⚠️ Colonnes avec >50% de données manquantes imputées :")
if len(sparse) > 0:
    for idx, row in sparse.iterrows():
        pct_miss = row['n_missing_before'] / row['nb_observations'] * 100
        print(f"   ⚠️ {row['colonne']} : {pct_miss:.1f}% manquantes imputées")
else:
    print(f"   ✅ Aucune colonne avec >50% de manquantes")

# ============================================================================
# 6️⃣ RECOMMANDATIONS
# ============================================================================
print("\n" + "█"*80)
print("6️⃣  RECOMMANDATIONS & CONCLUSION")
print("█"*80)

print(f"\n✅ Imputations réalisées avec succès :")
print(f"   • Total couvert : {pct_of_total:.2f}% des cellules vides")
print(f"   • Qualité globale : {'EXCELLENTE' if zscore_mean < 1 else 'BONNE' if zscore_mean < 2 else 'ACCEPTABLE' if zscore_mean < 3 else 'À VÉRIFIER'}")
print(f"   • Méthode : IterativeImputer (régression itérative)")

if len(anomalies) > 0:
    print(f"\n   ⚠️ ACTION : Vérifier les {len(anomalies)} colonne(s) anormale(s)")
    print(f"      → Considérer une suppression ou une re-imputation manuelle")

if len(sparse) > 0:
    print(f"\n   ⚠️ ACTION : {len(sparse)} colonne(s) très sparse (>50% manquantes)")
    print(f"      → Vérifier si ces colonnes sont essentielles pour le modèle")
    print(f"      → Considérer une suppression si redondante")

print(f"\n💾 Données imputées prêtes pour :")
print(f"   • Analyses statistiques")
print(f"   • Construction de modèles ML")
print(f"   • Visualisations multivariées")

print("\n" + "="*80)

In [ ]:
# ============================================================================
# 📊 VISUALISATIONS DE LA QUALITÉ DES IMPUTATIONS
# ============================================================================

print("\n" + "="*80)
print("📊 VISUALISATIONS DE LA QUALITÉ DES IMPUTATIONS".center(80))
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Évaluation de la Qualité des Imputations', fontsize=16, fontweight='bold')

# --- Graphique 1 : Distribution des Z-scores ---
imputed_cols_clean = imputation_report[imputation_report['n_imputed'] > 0].copy()
imputed_cols_clean['zscore_clean'] = imputed_cols_clean['imputed_zscore_abs_mean'].fillna(0)

ax1 = axes[0, 0]
ax1.hist(imputed_cols_clean['zscore_clean'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(1, color='green', linestyle='--', linewidth=2, label='Excellent (Z<1)')
ax1.axvline(2, color='orange', linestyle='--', linewidth=2, label='Acceptable (Z<2)')
ax1.axvline(3, color='red', linestyle='--', linewidth=2, label='Anomalie (Z>3)')
ax1.set_xlabel('Z-score absolu moyen', fontweight='bold')
ax1.set_ylabel('Nombre de colonnes', fontweight='bold')
ax1.set_title('Distribution des Z-scores d\'écart', fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# --- Graphique 2 : % de données imputées par colonne ---
ax2 = axes[0, 1]
top_15 = (
    imputation_report[imputation_report['n_imputed'] > 0]
    .sort_values(['pct_imputed', 'n_imputed'], ascending=False)
    .head(15)
    .copy()
)
colors = ['red' if x > 50 else 'orange' if x > 25 else 'steelblue' for x in top_15['pct_imputed']]
ax2.barh(range(len(top_15)), top_15['pct_imputed'], color=colors, edgecolor='black', alpha=0.7)
ax2.set_yticks(range(len(top_15)))
ax2.set_yticklabels([c[:25] for c in top_15['colonne']], fontsize=9)
ax2.invert_yaxis()
ax2.set_xlabel('% de données imputées', fontweight='bold')
ax2.set_title('Top 15 colonnes : Taux d\'imputation', fontweight='bold')
ax2.grid(alpha=0.3, axis='x')

# --- Graphique 3 : Écart absolu à la médiane ---
ax3 = axes[1, 0]
mad_values = imputed_cols_clean['imputed_abs_dev_from_original_median_mean'].dropna()
if len(mad_values) > 0:
    ax3.hist(mad_values, bins=20, color='seagreen', edgecolor='black', alpha=0.7)
    ax3.axvline(mad_values.mean(), color='red', linestyle='--', linewidth=2, label=f'Moyenne: {mad_values.mean():.2f}')
    ax3.set_xlabel('Écart absolu à la médiane', fontweight='bold')
    ax3.set_ylabel('Nombre de colonnes', fontweight='bold')
    ax3.set_title('Cohérence des imputations (MAD)', fontweight='bold')
    ax3.legend()
    ax3.grid(alpha=0.3)

# --- Graphique 4 : Couverture globale ---
ax4 = axes[1, 1]
categories = ['Avant\n(Manquantes)', 'Après\n(Manquantes)', 'Imputées\n(Couvertes)']
values = [total_missing_before, total_missing_after, total_imputed]
colors_pie = ['#ff6b6b', '#ffd93d', '#52b788']

if total_cells > 0:
    wedges, texts, autotexts = ax4.pie(
        [total_missing_before, total_imputed, total_cells - total_missing_before - total_imputed],
        labels=['Manquantes avant', 'Imputées', 'Complètes'],
        autopct='%1.1f%%',
        colors=['#ff6b6b', '#ffd93d', '#52b788'],
        startangle=90,
        textprops={'fontweight': 'bold'}
    )
else:
    ax4.text(0.5, 0.5, 'Aucune colonne analysée', ha='center', va='center', fontweight='bold')
ax4.set_title('Couverture globale des données', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Visualisations générées avec succès")

In [ ]:
# Afficher les colonnes avec z-score > 2
print("="*80)
print("Colonnes avec Z-score supérieur à 2")
print("="*80)

high_zscore = imputation_report[imputation_report['imputed_zscore_abs_mean'] > 2].sort_values('imputed_zscore_abs_mean', ascending=False)

if len(high_zscore) > 0:
    print(f"\n🔴 {len(high_zscore)} colonne(s) avec Z-score > 2:\n")
    for idx, row in high_zscore.iterrows():
        col_name = row['colonne']
        zscore = row['imputed_zscore_abs_mean']
        n_imputed = row['n_imputed']
        pct_imputed = row['pct_imputed']
        print(f"   • {col_name}")
        print(f"     └─ Z-score: {zscore:.3f} | Imputées: {int(n_imputed)} ({pct_imputed:.1f}%)")
else:
    print("\n✅ Aucune colonne avec Z-score > 2")

#### 1.3.2. Analyse de Sensibilité : Données Imputées vs Non-Imputées

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import ttest_ind, f_oneway, pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*100)
print("🔬 ANALYSE DE SENSIBILITÉ : DONNÉES IMPUTÉES vs NON-IMPUTÉES".center(100))
print("="*100)

# ============================================================================
# 1️⃣ CRÉATION DES DEUX DATASETS
# ============================================================================
print("\n" + "█"*100)
print("1️⃣  PRÉPARATION DES DATASETS POUR COMPARAISON")
print("█"*100)

# Dataset complet (avec imputées)
df_complete = df_after.copy()
print(f"✅ Dataset COMPLET (imputé) : {df_complete.shape[0]} rows × {df_complete.shape[1]} cols")
print(f"   Valeurs manquantes : {df_complete.isna().sum().sum()}")

# Dataset avec seulement données observées (non-imputées)
df_observed_only = df_before.copy()
print(f"✅ Dataset OBSERVÉ UNIQUEMENT : {df_observed_only.shape[0]} rows × {df_observed_only.shape[1]} cols")
print(f"   Valeurs manquantes : {df_observed_only.isna().sum().sum()}")

# Pour comparaison : on va sélectionner seulement les lignes/colonnes avec données complètes
# Identifier les lignes et colonnes complètes dans le dataset observé
complete_rows_mask = ~df_observed_only.isna().any(axis=1)
complete_cols = df_observed_only.columns[~df_observed_only.isna().any(axis=0)]

df_observed_complete = df_observed_only[complete_cols].dropna(axis=0)
df_complete_subset = df_complete[complete_cols]

print(f"\n📊 Sous-ensemble avec données COMPLÈTES (observé + imputé):")
print(f"   • Lignes complètes (sans NA) : {len(df_observed_complete)}")
print(f"   • Colonnes complètes (sans NA) : {len(complete_cols)}")

# ============================================================================
# 2️⃣ TESTS STATISTIQUES (T-tests)
# ============================================================================
print("\n" + "█"*100)
print("2️⃣  TESTS STATISTIQUES : T-Tests sur Variables Imputées")
print("█"*100)

# Récupérer les colonnes qui ont été imputées (n_imputed > 0)
imputed_cols_list = imputation_report[imputation_report['n_imputed'] > 0]['colonne'].tolist()
print(f"\n📊 Variables imputées trouvées dans le rapport : {len(imputed_cols_list)}")

# Vérifier quelles colonnes imputées sont présentes dans df_before et df_after
imputed_in_data = [col for col in imputed_cols_list if col in df_before.columns and col in df_after.columns]
print(f"📊 Variables imputées présentes dans les données : {len(imputed_in_data)}")
print(f"   Colonnes : {', '.join(imputed_in_data[:10])}{'...' if len(imputed_in_data) > 10 else ''}\n")

ttest_results = []
for col in imputed_in_data:
    # Données observées (non-imputées) avant imputation
    obs_data_before = df_before[col].dropna()
    # Toutes les données après imputation
    data_after = df_after[col].dropna()
    
    if len(obs_data_before) > 1 and len(data_after) > 1:
        t_stat, p_value = ttest_ind(obs_data_before, data_after)
        # Récupérer les infos d'imputation pour cette colonne
        imputation_info = imputation_report[imputation_report['colonne'] == col].iloc[0]
        ttest_results.append({
            'Variable': col,
            't_statistic': t_stat,
            'p_value': p_value,
            'Significatif': 'Oui (α=0.05)' if p_value < 0.05 else 'Non',
            'N_Imputées': int(imputation_info['n_imputed']),
            'Pct_Imputées': f"{imputation_info['pct_imputed']:.1f}%"
        })

ttest_df = pd.DataFrame(ttest_results)

if len(ttest_df) > 0:
    print("Comparaison Observé (avant) vs Après imputation sur variables IMPUTÉES :")
    print(ttest_df[['Variable', 't_statistic', 'p_value', 'Significatif', 'N_Imputées', 'Pct_Imputées']].to_string(index=False))
    
    sig_count = (ttest_df['p_value'] < 0.05).sum()
    print(f"\n📊 Résumé : {sig_count}/{len(ttest_df)} variables imputées avec différence significative")
    if sig_count == 0:
        print("   ✅ Excellent : Les distributions observées et imputées sont équivalentes !")
    elif sig_count <= len(ttest_df) * 0.1:
        print("   ✅ Bon : Très peu de différences significatives (<10%)")
    else:
        print(f"   ⚠️ À vérifier : {sig_count/len(ttest_df)*100:.1f}% de différences")
else:
    print("⚠️ Aucune variable imputée trouvée pour le test")
    print(f"   Debug : imputed_cols_list = {imputed_cols_list}")
    print(f"   Debug : Colonnes dans df_before : {list(df_before.columns[:10])}")
    print(f"   Debug : Colonnes dans df_after : {list(df_after.columns[:10])}")

print("\n" + "="*100)

In [ ]:
# ============================================================================
# 📊 VISUALISATION : DENSITÉS OBSERVÉ vs IMPUTÉ
# ============================================================================

print("\n" + "="*100)
print("📊 COMPARAISON VISUELLE : DENSITÉS DES DISTRIBUTIONS".center(100))
print("="*100)

# Sélectionner les 12 variables avec le plus grand pourcentage d'imputations
# Récupérer toutes les colonnes qui ont au moins une valeur imputée
imputed_cols_list = [col for col in df_before.columns if df_before[col].isna().sum() > 0]

# Calculer le % d'imputation pour chaque variable
imputation_pcts = {}
for col in imputed_cols_list:
    n_missing = df_before[col].isna().sum()
    pct_imputed = (n_missing / len(df_before)) * 100
    imputation_pcts[col] = pct_imputed

# Trier par pourcentage décroissant et prendre les 12 premières
vars_to_plot = sorted(imputation_pcts.keys(), key=lambda x: imputation_pcts[x], reverse=True)[:12]
n_vars = len(vars_to_plot)
n_cols = 3
n_rows = (n_vars + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
if n_vars == 1:
    axes = np.array([axes])
axes = axes.flatten()

print(f"\n🎨 Création de {n_vars} graphiques de densité pour les variables imputées...\n")

for idx, col in enumerate(vars_to_plot):
    ax = axes[idx]
    
    # Récupérer les données
    obs_data = df_before[col].dropna()
    after_data = df_after[col].dropna()
    
    # Plot densités pour les données observées et après imputation
    try:
        obs_data.plot.kde(ax=ax, color='steelblue', linewidth=2.5, label=f'Observé (n={len(obs_data)})', legend=True)
    except:
        # Si KDE échoue pour données observées, utiliser histogram
        obs_data.plot.hist(ax=ax, color='steelblue', alpha=0.5, label=f'Observé (n={len(obs_data)})', legend=True, bins=15)
    
    try:
        after_data.plot.kde(ax=ax, color='coral', linewidth=2.5, label=f'Après imputation (n={len(after_data)})', legend=True)
    except:
        # Si KDE échoue pour données après, utiliser histogram
        after_data.plot.hist(ax=ax, color='coral', alpha=0.5, label=f'Après imputation (n={len(after_data)})', legend=True, bins=15)
    
    # Récupérer le % d'imputation pour cette colonne
    pct_imp = imputation_pcts.get(col, 0)
    
    ax.set_title(f'{col}\n({pct_imp:.1f}% imputées)', fontweight='bold', fontsize=11)
    ax.set_xlabel('Valeur', fontweight='bold')
    ax.set_ylabel('Densité', fontweight='bold')
    ax.grid(alpha=0.2)
    ax.legend(loc='upper left', fontsize=9)

# Supprimer les axes vides
for idx in range(n_vars, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Superposition des Densités : Observé vs Imputé', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print(f"\n✅ Graphiques générés pour {n_vars} variables imputées")
if n_vars < len(imputation_pcts):
    print(f"   (Affichage des {n_vars} avec les plus hauts pourcentages d'imputation)")
print("\n" + "="*100)

In [ ]:
# === Export des données imputées (rapport qualité / exploratoire) en Excel ===
# ⚠️ IMPORTANT : df_after provient d'un ImputationPipeline fit() sur L'ENSEMBLE du dataframe
# (train + test + APP_NC confondus, avant toute séparation train/test). C'est volontaire et
# pertinent ICI, car cette section sert uniquement à évaluer la QUALITÉ de l'imputation
# (rapport, z-scores, analyse de sensibilité) sur l'ensemble de la cohorte.
#
# ⚠️ En revanche, CE FICHIER NE DOIT PAS être utilisé comme base d'une analyse en aval
# (ex. PCA) que vous comptez comparer/croiser avec les résultats du modèle de classification :
# les valeurs imputées de df_after diffèrent (légèrement) de celles réellement utilisées par
# le modèle, car celui-ci impute avec un imputeur fit UNIQUEMENT sur le train (voir section 2),
# puis appliqué (transform) au test et aux APP_NC, afin d'éviter toute fuite de données.
#
# ➜ Pour une PCA (ou toute analyse à mettre en regard des résultats du modèle), utilisez
# plutôt `df_imputed_no_leak`, exporté juste après la section 2 (cellule
# "Export des données imputées SANS fuite train/test"), qui reflète exactement les valeurs
# utilisées par le pipeline de classification.

import pandas as pd
import os

print("\n" + "="*70)
print("📊 EXPORT DES DONNÉES IMPUTÉES (RAPPORT QUALITÉ / EXPLORATOIRE, dataset entier) EN EXCEL")
print("="*70)

# Définir le chemin d'export
output_dir = '.'
output_filename = 'DataSet_Patients&Temoins_Imputed_ExploratoireGlobal.xlsx'
output_path = os.path.join(output_dir, output_filename)

# Vérifier que df_after existe (données imputées)
if 'df_after' in dir():
    try:
        # Exporter en Excel
        df_after.to_excel(output_path, sheet_name='Imputed Data (global fit)', index=False, engine='openpyxl')
        print(f"✅ Données imputées (fit global, exploratoire) exportées avec succès !")
        print(f"📁 Chemin : {output_path}")
        print(f"📈 Dimensions : {df_after.shape[0]} lignes × {df_after.shape[1]} colonnes")
        print(f"💾 Fichier sauvegardé : {output_filename}")
        print(f"ℹ️  Pour une PCA à comparer avec le modèle, utilisez plutôt df_imputed_no_leak (voir section 2).")
    except Exception as e:
        print(f"⚠️ Erreur lors de l'export : {e}")
else:
    print("⚠️ Erreur : df_after (données imputées) non trouvé dans l'environnement")
    print("Assurez-vous que l'imputation a été exécutée d'abord.")

print("="*70)


### 1.4. Préparation répartition des effectifs
Les opérations sont faites à partir d'une copie du dataframe initial (df)

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Prepare data
# First, separate unclassified samples
unclassified_mask = df['Phenotype_final'] == 'APP_NC'
unclassified_data = df[unclassified_mask].copy()
classified_data = df[~unclassified_mask].copy()

# Print est utile pour des verifications rapides
print(f"Classified samples: {len(classified_data)}")
print(f"Unclassified samples: {len(unclassified_data)}")
print(f"\nClassified target distribution:")
print(classified_data['Phenotype_final'].value_counts())

# Pour afficher le dataframe de façon lisible
summary = pd.DataFrame({
    "count": [len(classified_data), len(unclassified_data)]
}, index=["classified", "unclassified"])
display(summary)
display(classified_data["Phenotype_final"].value_counts().to_frame("count"))


Préparation pour la validation croisée de la sélection de variables. Variables créées :
- feature_cols
- X_classified et X_unclassified
- y_classified et y_unclassified
- X_train et X_test
- Y_train et y_test

Traitements : harmonisation des données encodées pour les variables catégorielles, séparation pour entraînement et test

In [ ]:

# Prepare features and target for classified data
feature_cols = [col for col in classified_data.columns 
                if col not in ['ID_CLAAP', 'Ville', 'Phenotype_final']] # Exclu les colonnes non pertinentes

X_classified = classified_data[feature_cols].copy() # (copie pour éviter les warnings et modifications involontaires)
y_classified = classified_data['Phenotype_final'].copy() # Vecteur cible

# Encode categorical variables
X_classified['Sexe'] = X_classified['Sexe'].map({'F': 0, 'H': 1})

# NSC_GREMOTs (1,2,3) -> s'assurer numérique
if 'NSC_GREMOTs' in X_classified.columns:
    X_classified['NSC_GREMOTs'] = pd.to_numeric(X_classified['NSC_GREMOTs'], errors='coerce')


# CRITICAL: Split data by selecting 5 samples from each class for test set

# Get class counts to see what we're working with
print("Class distribution before splitting:")
print(y_classified.value_counts())

# Initialize lists for train and test indices (listes vides)
test_indices = []
train_indices = []

# Ajout : Nombre de samples test par classe (spécifique à "Temoin")
n_test_per_class = {"Temoin": 24}  # 24 pour Temoin (pour faire 20% en test), 5 par défaut pour les autres

# For each class, select appropriate number of samples for test
for class_name in y_classified.unique():
    class_indices = y_classified[y_classified == class_name].index.tolist()
    
    # Nombre de test à sélectionner pour cette classe
    n_test = n_test_per_class.get(class_name, 5)  # 5 par défaut
    
    if len(class_indices) >= n_test:
        # Randomly select n_test for test
        np.random.seed(42)
        test_class_indices = np.random.choice(class_indices, n_test, replace=False)
        train_class_indices = [idx for idx in class_indices if idx not in test_class_indices]
    
    test_indices.extend(test_class_indices)
    train_indices.extend(train_class_indices)
    
    print(f"{class_name}: {len(class_indices)} total, {len(test_class_indices)} test, {len(train_class_indices)} train")

# Create train and test sets using the selected indices
X_train = X_classified.loc[train_indices]
X_test = X_classified.loc[test_indices]
Y_train = y_classified.loc[train_indices]
y_test = y_classified.loc[test_indices]

print(f"\nFinal test set distribution:")
print(y_test.value_counts())

print(f"\nData splits:")
print(f"Training: {len(X_train)} ({len(X_train)/len(X_classified)*100:.1f}%)")
print(f"Test: {len(X_test)} ({len(X_test)/len(X_classified)*100:.1f}%)")

# Prepare unclassified data with same preprocessing
X_unclassified = unclassified_data[feature_cols].copy()
X_unclassified['Sexe'] = X_unclassified['Sexe'].map({'F': 0, 'H': 1})

# Apply same encodings to unclassified
if 'NSC_GREMOTs' in X_unclassified.columns:
    X_unclassified['NSC_GREMOTs'] = pd.to_numeric(X_unclassified['NSC_GREMOTs'], errors='coerce')


y_unclassified = unclassified_data['Phenotype_final'].copy()


## 2. Preprocessing Pipeline (Fit only on training data)
Application du ImputationPipeline sur les dataframes créés plus haut (X_train, X_test). Résidualisation (garder la partie non-expliquée par les facteurs confondants, Sexe et Age).

Variables créées :
- X_train_imputed et X_test_imputed
- X_unclassified_imputed

Ajout par rapport au notebook d'origine :
- Extraction sous format Excel du dataset avec données imputées

In [ ]:
# CRITICAL: Fit preprocessing only on training data
print("Fitting preprocessing pipeline on training data only...")

# 1. Imputation (on réutilise l'imputeur entraîné sur les données d'entraînement)
# On relance le même imputer (même seed notamment) mais uniquement sur le train pour éviter le data leakage (fuite de données), les valeurs peuvent changer.
imputer = ImputationPipeline()
X_train_imputed = imputer.fit_transform(X_train)

X_test_imputed = imputer.transform(X_test)
X_unclassified_imputed = imputer.transform(X_unclassified)

print(f"\nFeatures after imputation: {X_train_imputed.shape[1]}")
print(f"Missing values after imputation (train): {X_train_imputed.isnull().sum().sum()}")
print(f"Missing values after imputation (test): {X_test_imputed.isnull().sum().sum()}")

# 2. Residualization (remove confounding effects)
# Veut dire ajuster une régression sur les variables confondantes, puis garder les résidus = la partie de la variable qui n'est pas expliquée par les confounders, ce qui reste après avoir enlevé l'effet des confounders.
confounding_vars = ['Age_G1', 'Sexe', 'NSC_GREMOTs']  # Liste des variables confondantes à ajuster
residualizer = Residualizer(confounders=confounding_vars, drop_confounders=True) # Va estimer l’effet des confounders sur chaque variable explicative et le retirer.
X_train_residualized = residualizer.fit_transform(X_train_imputed) # On ajuste le modèle de régression entre chaque feature (variable dépendante) et les confounders (prédicteurs), puis on remplace chaque valeur par son résidu (valeur réelle − valeur prédite par les confounders).
X_test_residualized = residualizer.transform(X_test_imputed)
X_unclassified_residualized = residualizer.transform(X_unclassified_imputed)

# Ces lignes commentées montrent l’alternative : ne pas faire de residualisation (utile pour des comparaisons/expérimentations).
#X_train_residualized = X_train_imputed
#X_test_residualized = X_test_imputed
#X_unclassified_residualized = X_unclassified_imputed

print(f"\nFeatures after residualization: {X_train_residualized.shape[1]}")
print(f"Removed confounders: {confounding_vars}")

### 2.bis. Export des données imputées SANS fuite train/test (pour PCA / analyses en aval)

Contrairement à `df_after` (section 1, imputeur fit sur l'ensemble du dataframe), le tableau
ci-dessous est reconstruit à partir de `X_train_imputed`, `X_test_imputed` et
`X_unclassified_imputed` — c'est-à-dire les valeurs imputées **réellement utilisées par le
modèle de classification**, où l'imputeur n'a été *fit* que sur le train, puis simplement
*appliqué* (`transform`) au test et aux APP_NC (voir §2 ci-dessus, conseil anti-fuite de données).

**C'est ce tableau qu'il faut utiliser pour une PCA** (ou toute analyse dont les résultats
seront mis en regard de ceux du modèle de classification), afin de garder des valeurs
cohérentes et sans fuite entre les deux analyses.

Variable créée : `df_imputed_no_leak`


In [ ]:
# === Export des données imputées SANS fuite train/test (pour PCA / analyses en aval) ===
import os
from datetime import datetime

# Réassembler les identifiants (ID_CLAAP, Ville, Phenotype_final) avec les valeurs imputées
# réellement utilisées par le pipeline (imputeur fit sur train uniquement, cf. cellule précédente)
id_cols = ['ID_CLAAP', 'Ville', 'Phenotype_final']

train_full = classified_data.loc[X_train_imputed.index, id_cols].join(X_train_imputed)
test_full = classified_data.loc[X_test_imputed.index, id_cols].join(X_test_imputed)
unclassified_full = unclassified_data.loc[X_unclassified_imputed.index, id_cols].join(X_unclassified_imputed)

# On ajoute une colonne indiquant le statut (utile pour distinguer les groupes dans la PCA si besoin)
train_full['Split_modele'] = 'Train'
test_full['Split_modele'] = 'Test'
unclassified_full['Split_modele'] = 'Unclassified (APP_NC)'

df_imputed_no_leak = pd.concat([train_full, test_full, unclassified_full]).sort_index()

print(f"✅ Jeu de données imputées SANS fuite reconstruit : "
      f"{df_imputed_no_leak.shape[0]} lignes x {df_imputed_no_leak.shape[1]} colonnes")
print(df_imputed_no_leak.groupby(['Split_modele', 'Phenotype_final']).size())

# Comparaison rapide avec df_after (optionnel, pour se rendre compte de l'écart)
# Ne compare que les colonnes réellement numériques des DEUX côtés (ex. 'Sexe' est resté en
# texte 'F'/'H' dans df_after, alors qu'il est encodé 0/1 dans df_imputed_no_leak -> à exclure)
if 'df_after' in dir():
    common_cols = [
        c for c in X_train_imputed.columns
        if c in df_after.columns
        and pd.api.types.is_numeric_dtype(df_imputed_no_leak[c])
        and pd.api.types.is_numeric_dtype(df_after[c])
    ]
    excluded_non_numeric = [c for c in X_train_imputed.columns
                             if c in df_after.columns and c not in common_cols]
    if excluded_non_numeric:
        print(f"ℹ️ Colonnes exclues de la comparaison (non numériques des deux côtés) : {excluded_non_numeric}")

    diff = (df_imputed_no_leak.loc[df_imputed_no_leak.index, common_cols]
            - df_after.loc[df_imputed_no_leak.index, common_cols])
    print(f"\nÉcart moyen absolu vs df_after (imputation globale) sur les colonnes communes : "
          f"{diff.abs().mean().mean():.3f}")

output_dir = '.'
output_filename = f"DataSet_Patients&Temoins_Imputed_no_leak_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
output_path = os.path.join(output_dir, output_filename)

try:
    df_imputed_no_leak.to_excel(output_path, sheet_name='Imputed Data (no leak)', index=False, engine='openpyxl')
    print(f"\n✅ Données imputées (sans fuite) exportées avec succès !")
    print(f"📁 Chemin : {output_path}")
    print(f"📈 Dimensions : {df_imputed_no_leak.shape[0]} lignes × {df_imputed_no_leak.shape[1]} colonnes")
    print(f"💾 Fichier sauvegardé : {output_filename}")
except Exception as e:
    print(f"⚠️ Erreur lors de l'export : {e}")


## 3. Nested Cross-Validation for Feature Selection
Traitements effectués : ajout de bruit Gaussien, suppression des variables redondantes.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class GaussianNoiseTransformer(BaseEstimator, TransformerMixin):
    """Add Gaussian noise to features"""
    
    def __init__(self, noise_level=0.01, random_state=42):
        self.noise_level = noise_level
        self.random_state = random_state
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        np.random.seed(self.random_state)
        noise = np.random.normal(0, self.noise_level, X.shape)
        return X + noise

class DropoutTransformer(BaseEstimator, TransformerMixin):
    """Randomly set some features to zero (dropout)"""
    
    def __init__(self, dropout_rate=0.1, random_state=42):
        self.dropout_rate = dropout_rate
        self.random_state = random_state
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        np.random.seed(self.random_state)
        mask = np.random.random(X.shape) > self.dropout_rate
        return X * mask
    
from sklearn.feature_selection import mutual_info_classif
import numpy as np
import pandas as pd

# Supprimer des features redondantes fortement corrélées en gardant, dans chaque groupe de corrélations, la feature la plus informative vis‑à‑vis de la cible.
def filter_redundant_features(X, y, threshold=0.9):
    """Remove redundant features based on mutual information + correlation clustering"""
    corr_matrix = X.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    to_drop = set()
    for col in upper.columns:
        correlated = upper.index[upper[col] > threshold].tolist()
        if correlated:
            correlated.append(col)
            # Pick the best feature by mutual information
            mi_scores = mutual_info_classif(X[correlated], y, discrete_features='auto')
            keep = correlated[np.argmax(mi_scores)]
            for f in correlated:
                if f != keep:
                    to_drop.add(f)
    return X.drop(columns=to_drop)


In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import PowerTransformer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier as KNN
#leaveoneout
from sklearn.model_selection import LeaveOneOut


from sklearn.model_selection import ShuffleSplit


Définition du Pipeline pour la validation croisée :
- Définition du nombre de dossiers
- Validation interne (inner) et validation externe (outer)
- Variables créées : outer_scores, selected_features_list, sfs_per_fold

In [ ]:
def nested_cv_feature_selection(X, y, n_outer_folds=20, n_inner_folds=5, plot=True, top_n=20, return_plot=False):
    """Perform nested cross-validation for unbiased feature selection and model evaluation

    Returns:
        outer_scores: list of weighted F1 for each outer fold
        selected_features_list: flat list of selected features across folds
        sfs_per_fold: list of SFS objects (one per outer fold)
        stability_df: pandas DataFrame summarizing how often each feature was selected across folds
        (optionally) fig: matplotlib Figure if return_plot=True
    """
    
    outer_cv = StratifiedKFold(n_splits=n_outer_folds, shuffle=True, random_state=42)
    inner_cv = LeaveOneOut()  # Use Leave-One-Out for inner CV to maximize feature selection
    #RepeatedStratifiedKFold(n_splits=n_inner_folds, n_repeats=5, random_state=42)
    
    outer_scores = []
    selected_features_list = []
    sfs_per_fold = []  # <--- here we will store the sfs objects
    
    
    #handle correlated here ; élimine les variables hautement corrélées
    X = filter_redundant_features(X, y, threshold=0.9)
    print(f"Features after redundancy filtering: {X.shape[1]}")
       
    # Boucle de validation croisée externe
    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        print(f"\nOuter fold {fold + 1}/{n_outer_folds}")
        
        # Split data for this outer fold
        X_train_outer = X.iloc[train_idx]
        X_test_outer = X.iloc[test_idx]
        y_train_outer = y.iloc[train_idx]
        y_test_outer = y.iloc[test_idx]
        
        # Inner CV for feature selection with SMOTE-enabled pipeline ; boucle de validation croisée interne
        # Stratégie de leave-one-out comme stratégie interne pour maximiser la sélection de features (chaque échantillon est laissé de côté une fois pour validation)
        pipeline = ImbPipeline([
            ('scaler', StandardScaler()),
            ('gaussian_noise', GaussianNoiseTransformer(noise_level=0.1, random_state=42)),
            #('dropout', DropoutTransformer(dropout_rate=0.1, random_state=42)),
            #('smote', SMOTE(random_state=42)),
            ('classifier', LogisticRegression(
                penalty='l1',
                C=0.1,
                solver='liblinear',
                random_state=42,
                max_iter=1000,
                class_weight='balanced'
            ))  #KNN(n_neighbors=9, n_jobs=-1, weights='distance', algorithm='auto', p=2)
            ])
        
        # Sequential Feature Selection (SFS) pour la sélection de features dans la boucle interne
        sfs = SFS(
            estimator=pipeline,
            k_features="parsimonious",    #parsimonious
            forward=False,
            floating=True,
            scoring='f1_weighted',
            cv=inner_cv,
            n_jobs=13,
            verbose=0
        )
        
        # Fit feature selector on outer training data
        sfs.fit(X_train_outer, y_train_outer)
        selected_features = list(sfs.k_feature_names_)
        selected_features_list.extend(selected_features)
        
        print(f"Selected {len(selected_features)} features, CV score: {sfs.k_score_:.3f}")
        print(f"Selected features: {selected_features}")
        
        # Train final model on selected features
        X_train_selected = X_train_outer[selected_features]
        X_test_selected = X_test_outer[selected_features]
        
        pipeline.fit(X_train_selected, y_train_outer)
        y_pred = pipeline.predict(X_test_selected)
        
        # Calculate outer fold score
        outer_score = f1_score(y_test_outer, y_pred, average='weighted')
        outer_scores.append(outer_score)
        
        print(f"Outer fold score: {outer_score:.3f}")
        # Save sfs for this fold
        sfs_per_fold.append(sfs)
    
    
    return outer_scores, selected_features_list, sfs_per_fold

Prochaine cellule exécute l'analyse.
- Variables créées : cv_scores, all_selected_features_
- Cellule de sauvegarde ajoutée à la suite

NB : exécution longue, stockage plus bas des résultats sous-forme de dictionnaire puis des variables sélectionnées dans une variable filtrant le dictionnaire

In [ ]:
# Run nested CV
print("Running nested cross-validation for unbiased evaluation...")
cv_scores, all_selected_features,sfs_per_fold = nested_cv_feature_selection(X_train_residualized, Y_train)

print(f"\nNested CV Results:")
print(f"Mean F1 Score: {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f}")
print(f"Individual fold scores: {[f'{score:.3f}' for score in cv_scores]}")

In [ ]:
import matplotlib.pyplot as plt
# Tracer pour chaque dossier la courbe d'évolution du score F1-wieghted fonction du nombre de features sélectionnées
plt.figure(figsize=(10, 6))
for fold_idx, sfs in enumerate(sfs_per_fold):
    # Extract scores and number of features
    num_features = [len(v['feature_idx']) for v in sfs.subsets_.values()]
    scores = [v['avg_score'] for v in sfs.subsets_.values()]
    
    # Sort in reverse order (max → min features)
    sorted_pairs = sorted(zip(num_features, scores), reverse=True)
    num_features_sorted, scores_sorted = zip(*sorted_pairs)
    
    # Plot
    plt.plot(num_features_sorted, scores_sorted, marker='o', label=f'Fold {fold_idx + 1}')

plt.xlabel("Number of features")
plt.ylabel("CV F1-weighted score")
plt.title("SFS score evolution across folds")
plt.grid(True)
plt.legend(title='Outer Fold', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. First, collect all (num_features, score) curves per fold
all_num_features = []
all_scores = []

for sfs in sfs_per_fold:
    num_features = [len(v['feature_idx']) for v in sfs.subsets_.values()]
    scores = [v['avg_score'] for v in sfs.subsets_.values()]
    
    # Sort in reverse
    sorted_pairs = sorted(zip(num_features, scores), reverse=True)
    num_features_sorted, scores_sorted = zip(*sorted_pairs)
    
    all_num_features.append(list(num_features_sorted))
    all_scores.append(list(scores_sorted))

# 2. Interpolate to common x-axis: number of features (max → min) (définir un axe commun)
max_num_features = max([max(fold_num_feat) for fold_num_feat in all_num_features])
common_num_features = list(range(max_num_features, 0, -1))  # [max, ..., 1]

# 3. Interpolate each fold’s scores to this common x
all_scores_interp = []

for num_feat, scores in zip(all_num_features, all_scores):
    # np.interp requires increasing x — reverse both
    interp_scores = np.interp(common_num_features[::-1], num_feat[::-1], scores[::-1])
    all_scores_interp.append(interp_scores[::-1])  # flip back

# 4. Compute mean and std across folds
all_scores_interp = np.array(all_scores_interp)  # shape (n_folds, n_features)
mean_scores = np.mean(all_scores_interp, axis=0)
std_scores = np.std(all_scores_interp, axis=0)

# 5. Plot mean ± std
plt.figure(figsize=(10, 6))
plt.plot(common_num_features, mean_scores, marker='o', color='blue', label='Mean CV F1-weighted score')
plt.fill_between(common_num_features, mean_scores - std_scores, mean_scores + std_scores,
                 color='blue', alpha=0.2, label='±1 std dev')

plt.xlabel("Number of features")
plt.ylabel("CV F1-weighted score , higher is better (from 0 to 1)")
plt.title("SFS aggregated score evolution (mean ± std across folds)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Refait plus bas après avoir regardé les fréquences de sélections des features
feature_counts = Counter(all_selected_features)

# Final model training on all selected features
stable_features = [feature for feature, count in feature_counts.items() if count >=10] # Garde uniquement les features sélectionnées dans au moins 10 folds
print(f"\nFinal selected features: {len(stable_features)}")
print(f"Selected features: {stable_features}")

# Me donne juste les features les plus stables, ici il n'y en a que 4.

In [ ]:
# Pour tracer l'histogramme des fréquences de sélection
from collections import Counter
feature_counts = Counter(all_selected_features)
print(feature_counts.most_common())

import matplotlib.pyplot as plt
names, counts = zip(*feature_counts.most_common())
plt.figure(figsize=(10,4))
plt.bar(range(len(names)), counts)
plt.xticks(range(len(names)), names, rotation=90)
plt.axhline(10, color='red', linestyle='--', label='threshold = 10')
plt.ylabel('Count across runs/folds')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Pour les fréquences de sélection avec seuils absolus et relatifs
# Imports
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# 1) Prépare les compteurs (si non déjà fait)
feature_counts = Counter(all_selected_features)  # all_selected_features doit exister
if len(feature_counts) == 0:
    print("Aucune feature trouvée dans `all_selected_features`.")
else:
    # 2) Paramètres de seuil (choisir l'un ou l'autre)
    abs_threshold = 10                   # seuil absolu (ex: >=10 fois)
    n_folds = len(sfs_per_fold) if 'sfs_per_fold' in globals() else None
    frac_threshold = 0.5                 # fraction des folds (ex: 0.5 => 50%)
    frac_threshold_count = int(np.ceil(frac_threshold * n_folds)) if n_folds else None

    # Choisir le seuil utilisé pour la ligne (définir 'threshold' ci-dessous)
    # threshold = abs_threshold
    threshold = frac_threshold_count if frac_threshold_count is not None else abs_threshold

    # 3) Prépare données triées
    names, counts = zip(*feature_counts.most_common())
    names = list(names)
    counts = np.array(list(counts))

    # 4) Plot horizontal (plus lisible pour noms de features)
    plt.figure(figsize=(10, max(4, len(names) * 0.25)))
    y_pos = np.arange(len(names))
    plt.barh(y_pos, counts, color='steelblue')
    plt.yticks(y_pos, names)
    plt.gca().invert_yaxis()  # mets la plus fréquente en haut
    plt.axvline(threshold, color='red', linestyle='--', label=f'seuil = {threshold}')
    plt.xlabel('Count across runs/folds')
    plt.title('Feature selection frequency across folds')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 5) Affiche les features stables selon le seuil
    stable_features = [f for f, c in feature_counts.items() if c >= threshold]
    print(f"\nSeuil utilisé = {threshold} ({'fractionnel ({}*n_folds)'.format(frac_threshold) if frac_threshold_count else 'absolu'})")
    print(f"Features stables ({len(stable_features)}): {stable_features}")

Stockage des variables sélectionnées sous la variable **stable_features**

In [ ]:
# On met ici les résultats précédents sous forme de dictionnaire pour ne pas avoir à refaire l'analyse complète
feature_counts_old=Counter({'Anamnestic_ITW_Prosody': 20,
         'Association_Photo_w_OralWord': 20,
        'Repetition_Sentence': 19,
        'Fluency_Semantic': 19,
        'Anamnestic_ITW_Lexic': 16,
        'Anamnestic_ITW_Understanding': 16,
        'Anamnestic_ITW_Pragmatics': 15,
        'Anamnestic_ITW_Total': 15,
        'Anamnestic_ITW_Intelligibility': 13,
        'Repetition_Words': 11,
        'Anamnestic_ITW_Syntax': 11,
        'Anamnestic_ITW_Attention': 11,
        'Denomination_Objects': 11,
        'Anamnestic_ITW_Info': 10,
        'Fluency_Verbs': 10,
        'Anamnestic_ITW_Fluidity': 10,
        'Execution_Commands': 7,
        'Fluency_Literal': 6,
        'Elaboration_Sentences': 6,
        'Narrative_Discourse_Lexic': 4,
        'Narrative_Discourse_Syntax': 4,
        'Narrative_Discourse_Actions': 2,
        'Narrative_Discourse_Recit': 1,
        'Reading_NonWords': 1,
        'Denomination_Verbs': 1,
        'Narrative_Discourse_Pragma': 1,
        'Writing_Dictation_Irreg_Words': 1,
        'Writing_Automatic': 1})


stable_features = [feature for feature, count in feature_counts_old.items() if count >= 10]
stable_features

## 4. Final Model Training and Evaluation

### 4.1. Entraînement du modèle de régression logistique

Paramétrage du modèle. Création des variables :
- final_model avec ImbPipeline

In [ ]:
# Pipeline d'apprentissage automatique pour la classification finale
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create pipeline: scale + classifier
final_model =  ImbPipeline([ # Crée un pipeline pour exécuter les étapes séquentiellement
            ('scaler', StandardScaler()), # Normalisation des données
            ('classifier', LogisticRegression( # Définit le classifieur
                penalty='l1', # Régularisation L1 (Lasso) pour réduire le surapprentissage
                C=0.2, # Force de régularisation (faible = plus forte régularisation)
                solver='liblinear', # Algorithme d'optimisation adapté à L1
                random_state=42,
                max_iter=1000,
                class_weight='balanced', # Gère les classes déséquilibrées, va aussi gerer le faite qu'il y ait plus de témoins
            )) 
            
             #LogisticRegression(solver='lbfgs', multi_class='multinomial')
             #DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced', min_samples_leaf=5))
        ])


Variables créées :
- X_train_slected et X_test_selected = avec les variables sélectionnées stockées dans stable_features
- X_unclassified_selected

In [ ]:
X_train_selected = X_train_residualized[stable_features]
X_test_selected = X_test_residualized[stable_features]
X_unclassified_selected = X_unclassified_residualized[stable_features]

# Ensure all data have the same columns (handle cases where features differ between train/test)
train_cols = set(X_train_selected.columns)
test_cols = set(X_test_selected.columns)
unclass_cols = set(X_unclassified_selected.columns)

# Get common columns across all splits
common_cols = sorted(list(train_cols & test_cols & unclass_cols))
print(f"Using {len(common_cols)} common features across all data splits")

# Filter to common columns to ensure consistency
X_train_selected = X_train_selected[common_cols]
X_test_selected = X_test_selected[common_cols]
X_unclassified_selected = X_unclassified_selected[common_cols]

Paramétrage de la validation croisée (cv) et exécution de la régression logistique

Modification : Changement d'une cellule pour avoir d'autre métriques en sortie mais méthode de régression identique

In [ ]:
# Cellule d'origine non-lancée. Lancée cellule suivante qui change pour récupérer toutes les prédictions LOO et avoir d'autres métriques des performances du modèle.

from sklearn.metrics import accuracy_score, f1_score

# Cross-validation training evaluation
print("Cross-Validation Training Performance:")
print("=" * 40)

# Perform cross-validation on training data
cv_train_scores = cross_val_score(final_model, X_train_selected, Y_train, 
                                  cv=LeaveOneOut(), scoring='accuracy', n_jobs=-1)

cv_train_f1_scores = cross_val_score(final_model, X_train_selected, Y_train, 
                                     cv=LeaveOneOut(), scoring='f1_weighted', n_jobs=-1)

print(f"Cross-Validation Accuracy: {cv_train_scores.mean():.3f} ± {cv_train_scores.std():.3f}")
print(f"Cross-Validation F1: {cv_train_f1_scores.mean():.3f} ± {cv_train_f1_scores.std():.3f}")
print(f"Individual CV accuracy scores: {[f'{score:.3f}' for score in cv_train_scores]}")
print(f"Individual CV F1 scores: {[f'{score:.3f}' for score in cv_train_f1_scores]}")

# Train final model on full training data
final_model.fit(X_train_selected, Y_train)

# Training performance
y_train_pred = final_model.predict(X_train_selected)
train_acc = accuracy_score(Y_train, y_train_pred)
train_f1 = f1_score(Y_train, y_train_pred, average='weighted')

# Test performance
y_test_pred = final_model.predict(X_test_selected)
test_acc = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f"\nFinal Model Performance:")
print(f"Training - Accuracy: {train_acc:.3f} | F1: {train_f1:.3f}")
print(f"Test     - Accuracy: {test_acc:.3f} | F1: {test_f1:.3f}")

# Generalization analysis
print(f"\nGeneralization Analysis:")
print(f"CV vs Test Accuracy gap: {cv_train_scores.mean() - test_acc:.3f}")
print(f"CV vs Test F1 gap: {cv_train_f1_scores.mean() - test_f1:.3f}")
print(f"Train vs Test Accuracy gap: {train_acc - test_acc:.3f}")
print(f"Train vs Test F1 gap: {train_f1 - test_f1:.3f}")

if (train_acc - test_acc) > 0.1:
    print("⚠️  Strong overfitting detected (accuracy gap > 0.1)")
elif (train_acc - test_acc) > 0.05:
    print("⚠️  Mild overfitting detected (accuracy gap > 0.05)")
else:
    print("✅ Good generalization (low overfitting)")


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report)

# 1. Cross-validation LOO : prédictions agrégées
y_cv_pred = cross_val_predict(final_model, X_train_selected, Y_train,
                              cv=LeaveOneOut(), n_jobs=-1)
cv_acc = accuracy_score(Y_train, y_cv_pred)
cv_f1  = f1_score(Y_train, y_cv_pred, average='weighted')

print("=== Cross-validation LOO ===")
print(f"Accuracy: {cv_acc:.3f} | F1 weighted: {cv_f1:.3f} | "
      f"F1 macro: {f1_score(Y_train, y_cv_pred, average='macro'):.3f} | "
      f"Balanced acc: {balanced_accuracy_score(Y_train, y_cv_pred):.3f}")
print(classification_report(Y_train, y_cv_pred, digits=3, zero_division=0))

# 2. Entraînement final sur tout le train
final_model.fit(X_train_selected, Y_train)

y_train_pred = final_model.predict(X_train_selected)
train_acc = accuracy_score(Y_train, y_train_pred)
train_f1  = f1_score(Y_train, y_train_pred, average='weighted')

# 3. Performance sur le test
y_test_pred = final_model.predict(X_test_selected)
test_acc = accuracy_score(y_test, y_test_pred)
test_f1  = f1_score(y_test, y_test_pred, average='weighted')

print("=== Test ===")
print(f"Balanced acc: {balanced_accuracy_score(y_test, y_test_pred):.3f}")
print(classification_report(y_test, y_test_pred, digits=3, zero_division=0))

# 4. Rapports par classe (nécessaires pour le tableau)
train_report = classification_report(Y_train, y_train_pred, output_dict=True, zero_division=0)
test_report  = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)

# 5. Analyse de généralisation
print(f"Train - Accuracy: {train_acc:.3f} | F1: {train_f1:.3f}")
print(f"Test  - Accuracy: {test_acc:.3f} | F1: {test_f1:.3f}")
print(f"CV vs Test Accuracy gap: {cv_acc - test_acc:.3f}")
print(f"CV vs Test F1 gap: {cv_f1 - test_f1:.3f}")
print(f"Train vs CV Accuracy gap: {train_acc - cv_acc:.3f}  (indicateur de surapprentissage)")

# 6. Tableau par classe
print(f"\n{'Classe':<15}{'Prec tr':<9}{'Prec te':<9}{'Rec tr':<9}{'Rec te':<9}{'F1 te':<9}{'n test'}")
for cls in final_model.classes_:
    k = str(cls)
    if k in train_report and k in test_report:
        tr, te = train_report[k], test_report[k]
        print(f"{k:<15}{tr['precision']:<9.3f}{te['precision']:<9.3f}"
              f"{tr['recall']:<9.3f}{te['recall']:<9.3f}{te['f1-score']:<9.3f}{int(te['support'])}")

In [ ]:
# IMPORTANT: Retrain the model to ensure it's fitted with current data columns
print("Retraining final model with current data columns...")
final_model.fit(X_train_selected, Y_train)
print(f"✓ Model retrained successfully on {X_train_selected.shape[0]} training samples with {X_train_selected.shape[1]} features")

In [ ]:
# Cellule d'origine, non utilisée car donnnait un problème d'affichage

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Get classifier
clf = final_model.named_steps['classifier']
assert clf.__class__.__name__ == "LogisticRegression", "Not a Logistic Regression model!"

# Get coefficients
coefs = clf.coef_  # shape: (n_classes, n_features)
class_names = clf.classes_

# Récupérer la liste complète des features originales (avant filtrage des inutilisés)
# Utiliser all_stable_features_original qui contient toutes les features sélectionnées
all_stable_features_original = stable_features.copy()  # Sauvegarder la liste complète avant filtrage
all_features = all_stable_features_original

# Créer une matrice complète avec tous les features (y compris les inutilisés)
complete_coefs = np.zeros((len(class_names), len(all_features)))

# Remplir avec les coefficients actuels
for i, feature in enumerate(all_features):
    if feature in stable_features:
        # Trouver l'index de cette feature dans stable_features
        idx_in_stable = stable_features.index(feature)
        complete_coefs[:, i] = coefs[:, idx_in_stable]
    # else: les coefficients restent 0 (features inutilisées)

# Create figure
plt.figure(figsize=(14, 10))

# Create a heatmap of coefficients
sns.heatmap(complete_coefs, 
            annot=True, 
            fmt='.2f',
            cmap='RdBu_r', 
            center=0,
            linewidths=0.5,
            xticklabels=all_features,
            yticklabels=class_names)

plt.title('Feature Importance by Class (Logistic Regression Coefficients)', fontsize=16)
plt.xlabel('Features', fontsize=14)
plt.ylabel('Class', fontsize=14)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Bug possible d'affichage corrigé par Claude. Donc code modifié.

clf = final_model.named_steps['classifier']

# En binaire, coef_ n'a qu'une ligne (celle de la classe "positive")
row_labels = clf.classes_ if clf.coef_.shape[0] > 1 else [clf.classes_[1]]
coef_df = pd.DataFrame(clf.coef_, index=row_labels, columns=X_train_selected.columns)

plt.figure(figsize=(14, 10))
sns.heatmap(coef_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5)
plt.title('Coefficients de la régression logistique par classe', fontsize=16)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

clf = final_model.named_steps['classifier']

# En binaire, coef_ n'a qu'une ligne (celle de la classe "positive")
row_labels = clf.classes_ if clf.coef_.shape[0] > 1 else [clf.classes_[1]]
coef_df = pd.DataFrame(clf.coef_, index=row_labels, columns=X_train_selected.columns)

# --- Renommage des groupes ---
group_names = {
    'Temoin': 'control',
    'APP_S': 'svPPA',
    'APP_NFA': 'nfvPPA',
    'APP_L': 'lvPPA',
}

# --- Renommage des variables ---
var_names = {
    'Anamnestic_ITW_Attention': 'Structured interview - Attention',
    'Anamnestic_ITW_Fluidity': 'Structured interview - Fluidity',
    'Anamnestic_ITW_Info': 'Structured interview - Informativeness',
    'Anamnestic_ITW_Intelligibility': 'Structured interview - Intelligibility',
    'Anamnestic_ITW_Lexic': 'Structured interview - Lexicon',
    'Anamnestic_ITW_Pragmatics': 'Structured interview - Pragmatics',
    'Anamnestic_ITW_Prosody': 'Structured interview - Prosody',
    'Anamnestic_ITW_Syntax': 'Structured interview - Syntax',
    'Anamnestic_ITW_Total': 'Structured interview - Overall',
    'Anamnestic_ITW_Understanding': 'Structured interview - Comprehension',
    'Association_Photo_w_OralWord': 'Oral word verification',
    'Denomination_Objects': 'Object naming',
    'Fluency_Semantic': 'Semantic fluency',
    'Fluency_Verbs': 'Verb fluency',
    'Repetition_Sentence': 'Sentence repetition',
    'Repetition_Words': 'Word repetition',
}

coef_df = coef_df.rename(index=group_names, columns=var_names)

# --- Graphique ---
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(coef_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Logistic regression coefficients by class', fontsize=16)

# Étiquettes en biais (45°), alignées sur leur case
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
plt.setp(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.svm import SVC

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(confusion_matrix(Y_train, y_train_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=final_model.classes_, yticklabels=final_model.classes_, ax=axes[0])
axes[0].set_title(f'Train Confusion Matrix\nAccuracy: {train_acc:.3f}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(confusion_matrix(y_test, y_test_pred), annot=True, fmt='d', cmap='Reds',
            xticklabels=final_model.classes_, yticklabels=final_model.classes_, ax=axes[1])
axes[1].set_title(f'Test Confusion Matrix\nAccuracy: {test_acc:.3f}')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.show()

# Per-class performance
train_report = classification_report(Y_train, y_train_pred, output_dict=True)
test_report = classification_report(y_test, y_test_pred, output_dict=True)

print(f"\nPer-Class F1 Analysis:")
print(f"{'Class':<15}{'Train F1':<10}{'Test F1':<10}{'Gap':<10}{'Test Samples'}")
for cls in final_model.classes_:
    if cls in train_report and cls in test_report:
        tr_f1 = train_report[cls]['f1-score']
        te_f1 = test_report[cls]['f1-score']
        gap = tr_f1 - te_f1
        support = test_report[cls]['support']
        print(f"{cls:<15}{tr_f1:<10.3f}{te_f1:<10.3f}{gap:<10.3f}{support}")

# Ajout proposer par Claude pour avoir un tableau plus complet avec précision, rappel et F1
print(f"\n{'Classe':<15}{'Prec tr':<9}{'Prec te':<9}{'Rec tr':<9}{'Rec te':<9}{'F1 te':<9}{'n test'}")
for cls in final_model.classes_:
    k = str(cls)  # les clés du report sont des chaînes
    if k in train_report and k in test_report:
        tr, te = train_report[k], test_report[k]
        print(f"{k:<15}{tr['precision']:<9.3f}{te['precision']:<9.3f}"
              f"{tr['recall']:<9.3f}{te['recall']:<9.3f}{te['f1-score']:<9.3f}{int(te['support'])}")


In [ ]:
from sklearn.svm import SVC

# --- Renommage des groupes ---
group_names = {
    'Temoin': 'control',
    'APP_S': 'svPPA',
    'APP_NFA': 'nfvPPA',
    'APP_L': 'lvPPA',
}
classes = final_model.classes_
display_labels = [group_names.get(str(c), str(c)) for c in classes]

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(confusion_matrix(Y_train, y_train_pred, labels=classes), annot=True, fmt='d', cmap='Blues',
            xticklabels=display_labels, yticklabels=display_labels, ax=axes[0])
axes[0].set_title(f'Train Confusion Matrix\nAccuracy: {train_acc:.3f}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(confusion_matrix(y_test, y_test_pred, labels=classes), annot=True, fmt='d', cmap='Reds',
            xticklabels=display_labels, yticklabels=display_labels, ax=axes[1])
axes[1].set_title(f'Test Confusion Matrix\nAccuracy: {test_acc:.3f}')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.show()

# Per-class performance
train_report = classification_report(Y_train, y_train_pred, output_dict=True)
test_report = classification_report(y_test, y_test_pred, output_dict=True)

print(f"\nPer-Class F1 Analysis:")
print(f"{'Class':<15}{'Train F1':<10}{'Test F1':<10}{'Gap':<10}{'Test Samples'}")
for cls in classes:
    k = str(cls)
    if k in train_report and k in test_report:
        tr_f1 = train_report[k]['f1-score']
        te_f1 = test_report[k]['f1-score']
        gap = tr_f1 - te_f1
        support = test_report[k]['support']
        name = group_names.get(k, k)
        print(f"{name:<15}{tr_f1:<10.3f}{te_f1:<10.3f}{gap:<10.3f}{int(support)}")

# Tableau complet : précision, rappel et F1
print(f"\n{'Class':<15}{'Prec tr':<9}{'Prec te':<9}{'Rec tr':<9}{'Rec te':<9}{'F1 te':<9}{'n test'}")
for cls in classes:
    k = str(cls)
    if k in train_report and k in test_report:
        tr, te = train_report[k], test_report[k]
        name = group_names.get(k, k)
        print(f"{name:<15}{tr['precision']:<9.3f}{te['precision']:<9.3f}"
              f"{tr['recall']:<9.3f}{te['recall']:<9.3f}{te['f1-score']:<9.3f}{int(te['support'])}")

Pour cellules suivantes, ajouts par rapport au code d'origine pour récupérer les ID des patients et les résultats correspondants

In [ ]:
# Récupérer les ID_CLAAP pour les patients et témoins d'entraînement
# Utiliser classified_data qui contient les indices correspondant à Y_train
train_ids_list = []
train_phenotypes_list = []

for idx in Y_train.index.tolist():
    try:
        # Récupérer l'ID_CLAAP de classified_data
        train_ids_list.append(classified_data.loc[idx, 'ID_CLAAP'])
        train_phenotypes_list.append(classified_data.loc[idx, 'Phenotype_final'])
    except (KeyError, TypeError):
        # Fallback si l'index n'existe pas
        train_ids_list.append(f"Index_{idx}")
        train_phenotypes_list.append("Unknown")

print(f"✓ Récupérés {len(train_ids_list)} ID_CLAAP pour les {len(Y_train)} patients/témoins d'entraînement")
print(f"  df initial shape: {df.shape}")
print(f"  classified_data shape: {classified_data.shape}")
print(f"  X_train_selected shape: {X_train_selected.shape}")

# Get prediction probabilities for training patients/controls with ID display
y_train_proba = final_model.predict_proba(X_train_selected)
train_max_probas = np.max(y_train_proba, axis=1)

# Create a DataFrame for easier manipulation and visualization
train_results_df = pd.DataFrame({
    'Index': Y_train.index.tolist(),
    'ID_CLAAP': train_ids_list,
    'Phenotype_True': train_phenotypes_list,
    'Prediction': y_train_pred,
    'True_Label': Y_train.values,
    'Max_Probability': train_max_probas,
    'Correct': y_train_pred == Y_train.values
})

# Create subplots: one per patient/control
n_samples = len(Y_train)
n_cols = 3

plt.figure(figsize=(15, max(8, n_samples//2)))

for i in range(n_samples):
    plt.subplot(max(1, n_samples//3) + min(1, n_samples%3), 3, i+1)
    
    # Get probabilities for each class for this patient/control
    probs = y_train_proba[i]
    
    # Plot horizontal bar chart
    bars = plt.barh(final_model.classes_, probs, color='skyblue', edgecolor='black')
    
    # Highlight predicted class and true class
    pred_idx = np.where(final_model.classes_ == y_train_pred[i])[0][0]
    true_idx = np.where(final_model.classes_ == Y_train.iloc[i])[0][0]
    
    # Color coding:
    # Correct prediction: green
    # Incorrect prediction: red for wrong, orange for true class
    if pred_idx == true_idx:
        bars[pred_idx].set_color('green')
    else:
        bars[pred_idx].set_color('red')       # Predicted class in red
        bars[true_idx].set_color('orange')    # True class in orange
    
    # Add probability values as text
    for j, v in enumerate(probs):
        plt.text(v + 0.01, j, f'{v:.2f}', va='center', fontsize=8)
    
    plt.xlim(0, 1.1)
    plt.ylim(-0.5, len(final_model.classes_) - 0.5)
    
    # Title with ID, phenotype, and accuracy indicator
    patient_id = train_ids_list[i]
    phenotype = train_phenotypes_list[i]
    correct_str = '✓' if y_train_pred[i] == Y_train.iloc[i] else '✗'
    
    plt.title(f'ID: {patient_id} | Phenotype: {phenotype}\nPred: {y_train_pred[i]} (prob: {train_max_probas[i]:.2f}) {correct_str}', 
              fontsize=9, fontweight='bold')
    plt.xlabel('Probability', fontsize=8)
    
    if i == 0:
        plt.ylabel('Class', fontsize=8)

plt.tight_layout()
plt.show()

# Print summary table
print("\n" + "="*80)
print("TRAINING SAMPLES SUMMARY")
print("="*80)
print(train_results_df.to_string(index=False))
print("="*80)

# Confidence distribution summary for training
print(f"\nTraining Confidence Summary:")
print(f"Mean: {train_max_probas.mean():.3f}, Std: {train_max_probas.std():.3f}, Min: {train_max_probas.min():.3f}, Max: {train_max_probas.max():.3f}")
high = np.sum(train_max_probas > 0.7)
mid = np.sum((train_max_probas > 0.5) & (train_max_probas <= 0.7))
low = np.sum(train_max_probas <= 0.5)
total = len(train_max_probas)
print(f"High (>0.7): {high} ({high/total*100:.1f}%)")
print(f"Medium (0.5-0.7): {mid} ({mid/total*100:.1f}%)")
print(f"Low (<=0.5): {low} ({low/total*100:.1f}%)")

# Accuracy by phenotype
print(f"\nAccuracy by Phenotype:")
correct_per_class = train_results_df.groupby('Phenotype_True')['Correct'].agg(['sum', 'count'])
correct_per_class.columns = ['Correct', 'Total']
correct_per_class['Accuracy'] = correct_per_class['Correct'] / correct_per_class['Total']
print(correct_per_class)

# Plot overall confidence distribution histogram for training
plt.figure(figsize=(10, 6))
plt.hist(train_max_probas, bins=15, alpha=0.7, edgecolor='black', color='lightgreen')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=2, label='Low/Medium threshold')
plt.axvline(0.7, color='red', linestyle='--', linewidth=2, label='Medium/High threshold')
plt.axvline(np.mean(train_max_probas), color='green', linestyle='-', linewidth=2, label=f'Mean: {np.mean(train_max_probas):.3f}')
plt.xlabel('Prediction Confidence', fontsize=11)
plt.ylabel('Sample Count', fontsize=11)
plt.title('Training Sample Confidence Distribution (Patients and Controls)', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Récupérer les ID_CLAAP pour les patients et témoins de test
# Utiliser classified_data qui contient les indices correspondant à y_test
test_ids_list = []
test_phenotypes_list = []

for idx in y_test.index.tolist():
    try:
        # Récupérer l'ID_CLAAP de classified_data
        test_ids_list.append(classified_data.loc[idx, 'ID_CLAAP'])
        test_phenotypes_list.append(classified_data.loc[idx, 'Phenotype_final'])
    except (KeyError, TypeError):
        # Fallback si l'index n'existe pas
        test_ids_list.append(f"Index_{idx}")
        test_phenotypes_list.append("Unknown")

print(f"✓ Récupérés {len(test_ids_list)} ID_CLAAP pour les {len(y_test)} patients/témoins de test")
print(f"  df shape: {df.shape}")
print(f"  classified_data shape: {classified_data.shape}")
print(f"  X_test_selected shape: {X_test_selected.shape}")

# Get prediction probabilities for test patients/controls with ID display
y_test_proba = final_model.predict_proba(X_test_selected)
test_max_probas = np.max(y_test_proba, axis=1)

# Create a DataFrame for easier manipulation and visualization
test_results_df = pd.DataFrame({
    'Index': y_test.index.tolist(),
    'ID_CLAAP': test_ids_list,
    'Phenotype_True': test_phenotypes_list,
    'Prediction': y_test_pred,
    'True_Label': y_test.values,
    'Max_Probability': test_max_probas,
    'Correct': y_test_pred == y_test.values
})

# Create subplots: one per patient/control
n_samples = len(y_test)
n_cols = 3

plt.figure(figsize=(15, max(8, n_samples//2)))

for i in range(n_samples):
    plt.subplot(max(1, n_samples//3) + min(1, n_samples%3), 3, i+1)
    
    # Get probabilities for each class for this patient/control
    probs = y_test_proba[i]
    
    # Plot horizontal bar chart
    bars = plt.barh(final_model.classes_, probs, color='skyblue', edgecolor='black')
    
    # Highlight predicted class and true class
    pred_idx = np.where(final_model.classes_ == y_test_pred[i])[0][0]
    true_idx = np.where(final_model.classes_ == y_test.iloc[i])[0][0]
    
    # Color coding:
    # Correct prediction: green
    # Incorrect prediction: red for wrong, orange for true class
    if pred_idx == true_idx:
        bars[pred_idx].set_color('green')
    else:
        bars[pred_idx].set_color('red')       # Predicted class in red
        bars[true_idx].set_color('orange')    # True class in orange
    
    # Add probability values as text
    for j, v in enumerate(probs):
        plt.text(v + 0.01, j, f'{v:.2f}', va='center', fontsize=8)
    
    plt.xlim(0, 1.1)
    plt.ylim(-0.5, len(final_model.classes_) - 0.5)
    
    # Title with ID, phenotype, and accuracy indicator
    patient_id = test_ids_list[i]
    phenotype = test_phenotypes_list[i]
    correct_str = '✓' if y_test_pred[i] == y_test.iloc[i] else '✗'
    
    plt.title(f'ID: {patient_id} | Phenotype: {phenotype}\nPred: {y_test_pred[i]} (prob: {test_max_probas[i]:.2f}) {correct_str}', 
              fontsize=9, fontweight='bold')
    plt.xlabel('Probability', fontsize=8)
    
    if i == 0:
        plt.ylabel('Class', fontsize=8)

plt.tight_layout()
plt.show()

# Print summary table
print("\n" + "="*80)
print("TEST SAMPLES SUMMARY")
print("="*80)
print(test_results_df.to_string(index=False))
print("="*80)

# Confidence distribution summary for test
print(f"\nTest Confidence Summary:")
print(f"Mean: {test_max_probas.mean():.3f}, Std: {test_max_probas.std():.3f}, Min: {test_max_probas.min():.3f}, Max: {test_max_probas.max():.3f}")
high = np.sum(test_max_probas > 0.7)
mid = np.sum((test_max_probas > 0.5) & (test_max_probas <= 0.7))
low = np.sum(test_max_probas <= 0.5)
total = len(test_max_probas)
print(f"High (>0.7): {high} ({high/total*100:.1f}%)")
print(f"Medium (0.5-0.7): {mid} ({mid/total*100:.1f}%)")
print(f"Low (<=0.5): {low} ({low/total*100:.1f}%)")

# Accuracy by phenotype
print(f"\nAccuracy by Phenotype:")
correct_per_class = test_results_df.groupby('Phenotype_True')['Correct'].agg(['sum', 'count'])
correct_per_class.columns = ['Correct', 'Total']
correct_per_class['Accuracy'] = correct_per_class['Correct'] / correct_per_class['Total']
print(correct_per_class)

# Plot overall confidence distribution histogram for test
plt.figure(figsize=(10, 6))
plt.hist(test_max_probas, bins=15, alpha=0.7, edgecolor='black', color='lightblue')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=2, label='Low/Medium threshold')
plt.axvline(0.7, color='red', linestyle='--', linewidth=2, label='Medium/High threshold')
plt.axvline(np.mean(test_max_probas), color='green', linestyle='-', linewidth=2, label=f'Mean: {np.mean(test_max_probas):.3f}')
plt.xlabel('Prediction Confidence', fontsize=11)
plt.ylabel('Sample Count', fontsize=11)
plt.title('Test Sample Confidence Distribution (Patients and Controls)', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Diagnostic: check which features are in X_test_selected vs what the model expects
print("Features in X_train_selected:", X_train_selected.columns.tolist())
print("Features in X_test_selected:", X_test_selected.columns.tolist())
print("Features in stable_features:", stable_features)

# Identify missing columns
train_cols_set = set(X_train_selected.columns)
test_cols_set = set(X_test_selected.columns)
model_expects = set(final_model.named_steps['scaler'].get_feature_names_out()) if hasattr(final_model.named_steps['scaler'], 'get_feature_names_out') else train_cols_set

missing_in_test = train_cols_set - test_cols_set
print(f"\nMissing columns in X_test_selected: {missing_in_test}")

# Add missing columns with 0 values (or median from training)
if len(missing_in_test) > 0:
    for col in missing_in_test:
        if col in X_train_selected.columns:
            # Fill missing columns with median from training data
            X_test_selected[col] = X_train_selected[col].median()
            print(f"Added missing column '{col}' with median value {X_train_selected[col].median()}")

# Ensure column order matches X_train_selected
X_test_selected = X_test_selected[X_train_selected.columns]
print(f"\nX_test_selected shape after alignment: {X_test_selected.shape}")
print(f"X_train_selected shape: {X_train_selected.shape}")

## 5. Prediction for Unclassified PPA

Ajouts effectués par rapport au code d'origine :
- Récupration des ID pour les unPPA
- Mise en forme de l'ensemble des résultats sous forme de tableau
- Graphiques pour représenter les résultats

### 5.1. Prédictions

In [ ]:
# Unclassified predictions using the LogisticRegression model
y_unclassified_pred = final_model.predict(X_unclassified_selected)
y_unclassified_proba = final_model.predict_proba(X_unclassified_selected)
max_probas = np.max(y_unclassified_proba, axis=1)
class_names = final_model.classes_

# Récupérer les ID_CLAAP pour les patients non classifiés
unclassified_ids_list = []
unclassified_phenotypes_list = []

for idx in X_unclassified.index.tolist():
    try:
        # Récupérer l'ID_CLAAP de unclassified_data
        unclassified_ids_list.append(unclassified_data.loc[idx, 'ID_CLAAP'])
        unclassified_phenotypes_list.append(unclassified_data.loc[idx, 'Phenotype_final'] if 'Phenotype_final' in unclassified_data.columns else 'Unknown')
    except (KeyError, TypeError):
        # Fallback si l'index n'existe pas
        unclassified_ids_list.append(f"Index_{idx}")
        unclassified_phenotypes_list.append("Unknown")

print(f"✓ Récupérés {len(unclassified_ids_list)} ID_CLAAP pour les {len(y_unclassified_pred)} patients/témoins non classifiés")
print(f"  df shape: {df.shape}")
print(f"  unclassified_data shape: {unclassified_data.shape}")
print(f"  X_unclassified_selected shape: {X_unclassified_selected.shape}")

# Create a DataFrame for easier manipulation and visualization
unclassified_results_df = pd.DataFrame({
    'Index': X_unclassified.index.tolist(),
    'ID_CLAAP': unclassified_ids_list,
    'Phenotype': unclassified_phenotypes_list,
    'Prediction': y_unclassified_pred,
    'Max_Probability': max_probas,
    'Confidence_Level': ['High' if p > 0.7 else 'Medium' if p > 0.5 else 'Low' for p in max_probas]
})

# Display predictions with confidence
print(f"\nUnclassified Sample Predictions:")
for i, (id_claap, pred, prob) in enumerate(zip(unclassified_ids_list, y_unclassified_pred, max_probas)):
    conf = "High" if prob > 0.7 else "Medium" if prob > 0.5 else "Low"
    print(f"Sample {i+1:2d}: ID={id_claap:<20} Pred={pred:<10} (confidence: {prob:.3f} - {conf})")

# Create subplots: one per unclassified patient
n_samples = len(y_unclassified_pred)
n_cols = 3

plt.figure(figsize=(15, max(8, n_samples//2)))

for i in range(n_samples):
    plt.subplot(max(1, n_samples//3) + min(1, n_samples%3), 3, i+1)
    
    # Get probabilities for each class for this patient
    probs = y_unclassified_proba[i]
    
    # Plot horizontal bar chart
    bars = plt.barh(class_names, probs, color='skyblue', edgecolor='black')
    
    # Highlight predicted class
    pred_idx = np.where(class_names == y_unclassified_pred[i])[0][0]
    bars[pred_idx].set_color('red')
    
    # Add probability values as text
    for j, v in enumerate(probs):
        plt.text(v + 0.01, j, f'{v:.2f}', va='center', fontsize=8)
    
    plt.xlim(0, 1.1)
    plt.ylim(-0.5, len(class_names) - 0.5)
    
    # Title with ID, confidence level, and probability
    patient_id = unclassified_ids_list[i]
    phenotype = unclassified_phenotypes_list[i]
    conf = "High" if max_probas[i] > 0.7 else "Medium" if max_probas[i] > 0.5 else "Low"
    
    plt.title(f'ID: {patient_id} | Phenotype: {phenotype}\nPred: {y_unclassified_pred[i]} (prob: {max_probas[i]:.2f} - {conf})', 
              fontsize=9, fontweight='bold')
    plt.xlabel('Probability', fontsize=8)
    
    if i == 0:
        plt.ylabel('Class', fontsize=8)

plt.tight_layout()
plt.show()

# Print summary table
print("\n" + "="*100)
print("UNCLASSIFIED SAMPLES SUMMARY")
print("="*100)
print(unclassified_results_df.to_string(index=False))
print("="*100)

# Confidence distribution summary
print(f"\nUnclassified Confidence Summary:")
print(f"Mean: {max_probas.mean():.3f}, Std: {max_probas.std():.3f}, Min: {max_probas.min():.3f}, Max: {max_probas.max():.3f}")
high = np.sum(max_probas > 0.7)
mid = np.sum((max_probas > 0.5) & (max_probas <= 0.7))
low = np.sum(max_probas <= 0.5)
total = len(max_probas)
print(f"High (>0.7): {high} ({high/total*100:.1f}%)")
print(f"Medium (0.5-0.7): {mid} ({mid/total*100:.1f}%)")
print(f"Low (<=0.5): {low} ({low/total*100:.1f}%)")

# Plot overall confidence distribution histogram
plt.figure(figsize=(10, 6))
plt.hist(max_probas, bins=15, alpha=0.7, edgecolor='black', color='lightyellow')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=2, label='Low/Medium threshold')
plt.axvline(0.7, color='red', linestyle='--', linewidth=2, label='Medium/High threshold')
plt.axvline(np.mean(max_probas), color='green', linestyle='-', linewidth=2, label=f'Mean: {np.mean(max_probas):.3f}')
plt.xlabel('Prediction Confidence', fontsize=11)
plt.ylabel('Sample Count', fontsize=11)
plt.title('Unclassified Sample Confidence Distribution', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2. Graphiques

In [ ]:
# ============================================================================
# 📊 MATRICE DE CONFUSION GLOBALE : TRAIN + TEST + UNCLASSIFIED (affichage en anglais)
# ============================================================================

# Correspondance noms internes -> noms affichés
label_map = {'APP_S': 'svPPA', 'APP_L': 'lvPPA', 'APP_NFA': 'nfvPPA',
             'Temoin': 'control', 'APP_NC': 'unPPA'}
disp = lambda c: label_map.get(str(c), str(c))

classes = list(final_model.classes_)
classes_disp = [disp(c) for c in classes]

def _rows(y_true, y_pred, prefix):
    """Une ligne par vraie classe : comptes des prédictions par classe prédite."""
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    return pd.DataFrame(cm, index=[f"{prefix} | {disp(c)}" for c in classes], columns=classes_disp)

cm_train = _rows(Y_train, y_train_pred, "Train")
cm_test  = _rows(y_test,  y_test_pred,  "Test")

# Ligne unique pour les APP non classifiées
unclass_counts = pd.Series(y_unclassified_pred).value_counts().reindex(classes, fill_value=0)
cm_unclass = pd.DataFrame([unclass_counts.values], index=[f"Unclassified | {disp('APP_NC')}"],
                          columns=classes_disp)

cm_all = pd.concat([cm_train, cm_test, cm_unclass])
cm_all = cm_all[cm_all.sum(axis=1) > 0]   # retirer les lignes vides

# Pourcentages par ligne pour la couleur
cm_pct = cm_all.div(cm_all.sum(axis=1), axis=0) * 100
annot = cm_all.astype(str) + "\n(" + cm_pct.round(0).astype(int).astype(str) + "%)"

fig, ax = plt.subplots(figsize=(1.8 * len(classes) + 4, 0.6 * len(cm_all) + 2))
sns.heatmap(cm_pct, annot=annot, fmt='', cmap='Blues', vmin=0, vmax=100,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Row percentage (%)'}, ax=ax)

# Lignes de séparation entre Train / Test / Unclassified
prefixes = [r.split(' | ')[0] for r in cm_all.index]
for i in range(1, len(prefixes)):
    if prefixes[i] != prefixes[i - 1]:
        ax.axhline(i, color='black', linewidth=2.5)

# Encadrer la diagonale (bonnes prédictions) pour Train et Test
for i, row in enumerate(cm_all.index):
    set_name, true_cls = row.split(' | ')
    if set_name in ('Train', 'Test') and true_cls in classes_disp:
        j = classes_disp.index(true_cls)
        ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='green', lw=2.5))

ax.set_title(f"Confusion matrix – Train (acc. {train_acc:.2f}), Test (acc. {test_acc:.2f}) "
             f"and {disp('APP_NC')} (n={len(y_unclassified_pred)})",
             fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted class', fontsize=11)
ax.set_ylabel('Dataset | True class', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 🔥 HEATMAP DES PROBABILITÉS PRÉDITES – APP NON CLASSIFIÉES (affichage en anglais)
# ============================================================================

# Correspondance noms internes -> noms affichés
label_map = {'APP_S': 'svPPA', 'APP_L': 'lvPPA', 'APP_NFA': 'nfvPPA',
             'Temoin': 'control', 'APP_NC': 'unPPA'}
disp = lambda c: label_map.get(str(c), str(c))

class_list = list(class_names)
class_list_disp = [disp(c) for c in class_list]

proba_unclass_df = pd.DataFrame(y_unclassified_proba,
                                index=[str(i) for i in unclassified_ids_list],
                                columns=class_list)
proba_unclass_df['_pred'] = y_unclassified_pred
proba_unclass_df['_max'] = max_probas

# Tri : regrouper par classe prédite, puis confiance décroissante
order_map = {c: k for k, c in enumerate(class_list)}
proba_unclass_df['_order'] = proba_unclass_df['_pred'].map(order_map)
proba_unclass_df = proba_unclass_df.sort_values(['_order', '_max'], ascending=[True, False])

heat = proba_unclass_df[class_list].copy()
heat.columns = class_list_disp
conf_lbl = ['High' if p > 0.7 else 'Medium' if p > 0.5 else 'Low' for p in proba_unclass_df['_max']]
heat.index = [f"{idx}  ({c})" for idx, c in zip(heat.index, conf_lbl)]

fig, ax = plt.subplots(figsize=(1.6 * len(class_list) + 4, 0.4 * len(heat) + 2))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Predicted probability'}, ax=ax)

# Encadrer la classe prédite
for i, pred in enumerate(proba_unclass_df['_pred']):
    j = class_list.index(pred)
    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='black', lw=2))

# Séparer les groupes de classe prédite
preds = proba_unclass_df['_pred'].tolist()
for i in range(1, len(preds)):
    if preds[i] != preds[i - 1]:
        ax.axhline(i, color='black', linewidth=2)

ax.set_title(f"Predicted class probabilities – {disp('APP_NC')} patients (n={len(heat)})",
             fontsize=13, fontweight='bold')
ax.set_xlabel('Diagnostic class', fontsize=11)
ax.set_ylabel('Patient (ID_CLAAP) – confidence level', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 🔥 HEATMAP DES PROBABILITÉS PRÉDITES – APP NON CLASSIFIÉES (affichage en anglais + P1, P2, etc.)
# ============================================================================

# Correspondance noms internes -> noms affichés
label_map = {'APP_S': 'svPPA', 'APP_L': 'lvPPA', 'APP_NFA': 'nfvPPA',
             'Temoin': 'control', 'APP_NC': 'unPPA'}
disp = lambda c: label_map.get(str(c), str(c))

# Anonymisation : ID_CLAAP triés par ordre alphabétique -> P1, P2, ...
ids_sorted = sorted(str(i) for i in unclassified_ids_list)
patient_code = {pid: f"P{k}" for k, pid in enumerate(ids_sorted, start=1)}

# Vérification de la correspondance
print(pd.DataFrame({'ID_CLAAP': list(patient_code.keys()),
                    'Code': list(patient_code.values())}).to_string(index=False))

class_list = list(class_names)
class_list_disp = [disp(c) for c in class_list]

proba_unclass_df = pd.DataFrame(y_unclassified_proba,
                                index=[str(i) for i in unclassified_ids_list],
                                columns=class_list)
proba_unclass_df['_pred'] = y_unclassified_pred
proba_unclass_df['_max'] = max_probas

# Tri : regrouper par classe prédite, puis confiance décroissante
order_map = {c: k for k, c in enumerate(class_list)}
proba_unclass_df['_order'] = proba_unclass_df['_pred'].map(order_map)
proba_unclass_df = proba_unclass_df.sort_values(['_order', '_max'], ascending=[True, False])

heat = proba_unclass_df[class_list].copy()
heat.columns = class_list_disp
conf_lbl = ['High' if p > 0.7 else 'Medium' if p > 0.5 else 'Low' for p in proba_unclass_df['_max']]
heat.index = [f"{patient_code[idx]}  ({c})" for idx, c in zip(heat.index, conf_lbl)]

fig, ax = plt.subplots(figsize=(1.6 * len(class_list) + 4, 0.4 * len(heat) + 2))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Predicted probability'}, ax=ax)

# Encadrer la classe prédite
for i, pred in enumerate(proba_unclass_df['_pred']):
    j = class_list.index(pred)
    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='black', lw=2))

# Séparer les groupes de classe prédite
preds = proba_unclass_df['_pred'].tolist()
for i in range(1, len(preds)):
    if preds[i] != preds[i - 1]:
        ax.axhline(i, color='black', linewidth=2)

ax.set_title(f"Predicted class probabilities – {disp('APP_NC')} patients (n={len(heat)})",
             fontsize=13, fontweight='bold')
ax.set_xlabel('Diagnostic class', fontsize=11)
ax.set_ylabel('Patient – confidence level', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 📋 HEATMAP RÉCAPITULATIVE – unPPA : classe prédite × niveau de confiance
# ============================================================================
# Lignes   = classe prédite (+ ligne "All" = total)
# Colonnes = niveau de confiance (Low ≤ 0.5 < Medium ≤ 0.7 < High) (+ colonne "Total")
# Couleur  = nombre de patients
# Texte    = n patients, puis [min – max] de la probabilité de la classe prédite

label_map = {'APP_S': 'svPPA', 'APP_L': 'lvPPA', 'APP_NFA': 'nfvPPA',
             'Temoin': 'control', 'APP_NC': 'unPPA'}
disp = lambda c: label_map.get(str(c), str(c))

levels = ['Low', 'Medium', 'High']
cols = levels + ['Total']

summ = pd.DataFrame({
    'pred': [disp(p) for p in y_unclassified_pred],
    'proba': max_probas,
    'level': ['High' if p > 0.7 else 'Medium' if p > 0.5 else 'Low' for p in max_probas]
})

rows = [disp(c) for c in class_names] + ['All']
counts = pd.DataFrame(0, index=rows, columns=cols)
annot = pd.DataFrame('', index=rows, columns=cols)

def _cell(sub):
    """Texte d'une case : n et [min – max] des probabilités."""
    if len(sub) == 0:
        return 0, '–'
    return len(sub), f"n = {len(sub)}\n[{sub['proba'].min():.2f} – {sub['proba'].max():.2f}]"

for r in rows:
    sub_r = summ if r == 'All' else summ[summ['pred'] == r]
    for c in cols:
        sub = sub_r if c == 'Total' else sub_r[sub_r['level'] == c]
        counts.loc[r, c], annot.loc[r, c] = _cell(sub)

fig, ax = plt.subplots(figsize=(10, 0.9 * len(rows) + 2))
sns.heatmap(counts, annot=annot, fmt='', cmap='Purples', vmin=0,
            vmax=counts.loc[rows[:-1], levels].values.max(),  # échelle basée sur les cases détaillées
            linewidths=0.5, linecolor='white',
            annot_kws={'fontsize': 10},
            cbar_kws={'label': 'Number of patients'}, ax=ax)

# Séparer la ligne "All" et la colonne "Total"
ax.axhline(len(rows) - 1, color='black', linewidth=2.5)
ax.axvline(len(levels), color='black', linewidth=2.5)

ax.set_title(f"Summary of predictions – {disp('APP_NC')} patients (n={len(summ)})\n"
             "Cells: number of patients and [min – max] probability of the predicted class",
             fontsize=12, fontweight='bold')
ax.set_xlabel('Confidence level  (Low ≤ 0.5 < Medium ≤ 0.7 < High)', fontsize=11)
ax.set_ylabel('Predicted class', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 📋 HEATMAP RÉCAPITULATIVE – unPPA : classe prédite × niveau de confiance
# ============================================================================
# Lignes   = classe prédite
# Colonnes = niveau de confiance (Low ≤ 0.5 < Medium ≤ 0.7 < High)
# Couleur  = nombre de patients
# Texte    = n patients, puis [min – max] de la probabilité de la classe prédite

label_map = {'APP_S': 'svPPA', 'APP_L': 'lvPPA', 'APP_NFA': 'nfvPPA',
             'Temoin': 'control', 'APP_NC': 'unPPA'}
disp = lambda c: label_map.get(str(c), str(c))

levels = ['Low', 'Medium', 'High']

summ = pd.DataFrame({
    'pred': [disp(p) for p in y_unclassified_pred],
    'proba': max_probas,
    'level': ['High' if p > 0.7 else 'Medium' if p > 0.5 else 'Low' for p in max_probas]
})

rows = [disp(c) for c in class_names]
counts = pd.DataFrame(0, index=rows, columns=levels)
annot = pd.DataFrame('', index=rows, columns=levels)

for r in rows:
    for c in levels:
        sub = summ[(summ['pred'] == r) & (summ['level'] == c)]
        if len(sub) == 0:
            annot.loc[r, c] = '–'
        else:
            counts.loc[r, c] = len(sub)
            annot.loc[r, c] = f"n = {len(sub)}\n[{sub['proba'].min():.2f} – {sub['proba'].max():.2f}]"

fig, ax = plt.subplots(figsize=(8, 0.9 * len(rows) + 2))
sns.heatmap(counts, annot=annot, fmt='', cmap='Purples', vmin=0,
            linewidths=0.5, linecolor='white',
            annot_kws={'fontsize': 10},
            cbar_kws={'label': 'Number of patients'}, ax=ax)

ax.set_title(f"Summary of predictions – {disp('APP_NC')} patients (n={len(summ)})\n"
             "Cells: number of patients and [min – max] probability of the predicted class",
             fontsize=12, fontweight='bold')
ax.set_xlabel('Confidence level  (Low ≤ 0.5 < Medium ≤ 0.7 < High)', fontsize=11)
ax.set_ylabel('Predicted class', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### 5.3. Récupération des résultats

Récupération des résultats des prédictions en tableur

In [ ]:
# Créer un dataframe récapitulatif avec tous les résultats
print("=" * 100)
print("CRÉATION DU DATAFRAME RÉCAPITULATIF")
print("=" * 100)

# Initialiser listes pour le dataframe
all_ids = []
all_phenotypes = []
all_status = []
all_probs = []

# Ajouter les données de training
print(f"\nAjout de {len(train_ids_list)} échantillons TRAIN...")
for i, (id_claap, phenotype) in enumerate(zip(train_ids_list, train_phenotypes_list)):
    all_ids.append(id_claap)
    all_phenotypes.append(phenotype)
    all_status.append('Train')
    all_probs.append(y_train_proba[i])

# Ajouter les données de test
print(f"Ajout de {len(test_ids_list)} échantillons TEST...")
for i, (id_claap, phenotype) in enumerate(zip(test_ids_list, test_phenotypes_list)):
    all_ids.append(id_claap)
    all_phenotypes.append(phenotype)
    all_status.append('Test')
    all_probs.append(y_test_proba[i])

# Ajouter les données non classifiées
print(f"Ajout de {len(unclassified_ids_list)} échantillons UNCLASSIFIED...")
for i, (id_claap, phenotype) in enumerate(zip(unclassified_ids_list, unclassified_phenotypes_list)):
    all_ids.append(id_claap)
    all_phenotypes.append(phenotype)
    all_status.append('Unclassified')
    all_probs.append(y_unclassified_proba[i])

# Créer le dataframe avec les colonnes de probabilités pour chaque classe
probs_dict = {label: [] for label in final_model.classes_}
phenotypes_modele = []
niv_conf_list = []

for probs in all_probs:
    for j, label in enumerate(final_model.classes_):
        probs_dict[label].append(probs[j])
    
    # Obtenir le label prédit (probabilité maximale)
    max_idx = np.argmax(probs)
    phenotypes_modele.append(final_model.classes_[max_idx])
    
    # Obtenir le niveau de confiance
    max_prob = probs[max_idx]
    if max_prob > 0.7:
        niv_conf_list.append('High')
    elif max_prob > 0.5:
        niv_conf_list.append('Medium')
    else:
        niv_conf_list.append('Low')

# Construire le dataframe final
results_summary_df = pd.DataFrame({
    'ID_CLAAP': all_ids,
    'Phenotype_final': all_phenotypes,
    'Statut_modele': all_status
})

# Ajouter les colonnes de probabilités
for label in final_model.classes_:
    results_summary_df[label] = probs_dict[label]

# Ajouter les colonnes Phenotype_modele et Niv_conf
results_summary_df['Phenotype_modele'] = phenotypes_modele
results_summary_df['Niv_conf'] = niv_conf_list

print(f"\n✓ Dataframe récapitulatif créé avec succès !")
print(f"  Dimensions: {results_summary_df.shape}")
print(f"  Nbr Train: {len([s for s in all_status if s == 'Train'])}")
print(f"  Nbr Test: {len([s for s in all_status if s == 'Test'])}")
print(f"  Nbr Unclassified: {len([s for s in all_status if s == 'Unclassified'])}")
print(f"\nAperçu du dataframe:")
print(results_summary_df.head(10))
print(f"\n...")
print(results_summary_df.tail(10))

In [ ]:
# Exporter le dataframe en fichier xlsx
import os
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from datetime import datetime

# Définir le chemin de export
output_filename = f"Resultats_Predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
output_path = os.path.join(os.getcwd(), output_filename)

print(f"\nExport en cours...")
print(f"Fichier: {output_filename}")

# Exporter le dataframe
results_summary_df.to_excel(output_path, sheet_name='Resultats', index=False)

print(f"✓ Fichier exporté avec succès!")
print(f"  Chemin complet: {output_path}")
print(f"  Taille: {len(results_summary_df)} lignes")

# Ajouter de la mise en forme Excel
from openpyxl import load_workbook
wb = load_workbook(output_path)
ws = wb.active

# Style pour l'en-tête
header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
header_font = Font(bold=True, color="FFFFFF")

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Ajuster les largeurs de colonnes
for column in ws.columns:
    max_length = 0
    column_letter = column[0].column_letter
    for cell in column:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = min(max_length + 2, 50)
    ws.column_dimensions[column_letter].width = adjusted_width

# Format les probabilités avec 3 décimales
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for col_idx in range(4, ws.max_column + 1):  # Colonnes de probabilités (à partir de la 4ème)
        cell = row[col_idx - 1]
        if cell.value is not None and isinstance(cell.value, (int, float)):
            cell.number_format = '0.000'

wb.save(output_path)
print(f"✓ Mise en forme appliquée et fichier sauvegardé!")